In [ ]:
import pandas as pd
import numpy as np
import json
import os
import math
import calendar
from datetime import datetime, date

# =============================================================
#  Smart APS V11-DEMAND  —  Maximum Parts, Demand-Aware
# =============================================================

# =============================================================
# SECTION 1 — DAILY SETTINGS
# =============================================================

PLANNING_DATE = date(2026, 4, 9)
INDENT_MONTH  = date(2026, 4, 1)

# =============================================================
# SECTION 2 — PARAMETERS
# =============================================================

AVAILABLE_HOURS          = 22
AVAILABLE_HOURS_EXTENDED = 23
MIN_RUN_HOURS            = 4
MACHINE_STATE_FILE       = "machine_state.json"

MIN_DAILY_INDENT         = 150
MIN_INDENT_HOURS         = 4.0

SAFETY_DAYS              = 3
TARGET_DAYS              = 5

OPD_SCENARIO_0 = 3.0
OPD_SCENARIO_1 = 3.0
OPD_SCENARIO_2 = 4.0
OPD_SCENARIO_3 = 5.0

W_URGENCY  = 0.55
W_CATEGORY = 0.25
W_INDENT   = 0.20

UTIL_TARGET_PCT  = 90.0
COLOR_PURGE_HRS  = 10 / 60.0

RUNNER_PRIORITY_DAYS = 2.0

MAX_DAILY_CO             = 25

FORWARD_LOOK_DAYS        = 7

TERMINAL_THRESHOLD = {
    "Runner":   1.0,
    "Repeater": 0.5,
    "Stranger": 0.25,
}

STRATEGIC_BUFFER_DAYS        = 7
STRATEGIC_PRIORITY_DISCOUNT  = 0.5
EFFICIENCY_MODE_UTIL_FLOOR   = 95.0

ABSOLUTE_MAX_DAYS            = 15

# =============================================================
# SECTION 3 — FILE PATHS
# =============================================================

book_path       = "C:/Users/Ex0164/Book1.xlsx"
matrix_path     = "C:/Users/Ex0164/Important codes/compatibility_matrix.xlsx"
changeover_path = "D:/Tushar/TOOL FIX/Unique_Machines_vt.xlsx"
terminal_path   = "C:/Users/Ex0164/Important codes/terminals and raw marterial - vt.xlsx"
demand_path     = "C:/Users/Ex0164/Demand.xlsx"
output_path     = f"Smart_APS_V11_Plan_{PLANNING_DATE.strftime('%Y%m%d')}.xlsx"

# Demand file expected columns: "Part", "Daily_Demand"
# Daily demand is used directly — no division by working days
DEMAND_SHEET_NAME  = "VT_Demand"      # sheet name inside demand_path
DEMAND_PART_COL    = "Part"
DEMAND_DAILY_COL   = "Daily_Demand"   # <-- CHANGED: column now holds DAILY demand directly

# =============================================================
# SECTION 4 — WORKING DAYS
# =============================================================

def compute_working_days(ref_date):
    year  = ref_date.year
    month = ref_date.month
    total = calendar.monthrange(year, month)[1]
    sundays = sum(
        1 for d in range(1, total + 1)
        if date(year, month, d).weekday() == 6
    )
    return total - sundays, total, sundays

WORKING_DAYS, TOTAL_DAYS, SUNDAY_COUNT = compute_working_days(INDENT_MONTH)

print(f"\n{'='*65}")
print(f"  Smart APS V11-DEMAND  —  Maximum Parts, Demand-Aware")
print(f"  Planning date  : {PLANNING_DATE}")
print(f"  Indent month   : {INDENT_MONTH.strftime('%B %Y')}")
print(f"  Working days   : {WORKING_DAYS}  ({TOTAL_DAYS} days - {SUNDAY_COUNT} Sundays)")
print(f"  Safety floor   : {SAFETY_DAYS} days  |  Target ceiling : {TARGET_DAYS} days")
print(f"  Runner priority: inv < {RUNNER_PRIORITY_DAYS}x daily")
print(f"  Max daily CO   : {MAX_DAILY_CO}")
print(f"  Forward look   : {FORWARD_LOOK_DAYS} days (advisory)")
print(f"{'='*65}\n")

# =============================================================
# SECTION 5 — LOAD DATA
# =============================================================

print("Loading data...")
vt_parts_raw         = pd.read_excel(book_path,       sheet_name="VT")
vt_matrix            = pd.read_excel(matrix_path,     sheet_name="VT_Matrix")
vt_co_raw            = pd.read_excel(changeover_path, sheet_name="VT_Changeover")
vt_machine_count_raw = pd.read_excel(matrix_path,     sheet_name="VT_Machine_Part_Count")

try:
    vt_fixed_raw = pd.read_excel(matrix_path, sheet_name="VT_Fixed")
    print(f"  VT_Fixed sheet loaded  ({len(vt_fixed_raw)} rows)")
except Exception as _fe:
    vt_fixed_raw = None
    print(f"  WARNING: VT_Fixed sheet not found ({_fe})")

try:
    vt_terminals_raw      = pd.read_excel(terminal_path, sheet_name="VT_Terminals")
    vt_terminal_avail_raw = pd.read_excel(terminal_path, sheet_name="VT_Terminal_Inventory")
    print(f"  Terminal data loaded from  : {terminal_path}")
except FileNotFoundError:
    vt_terminals_raw      = None
    vt_terminal_avail_raw = None
    print(f"  WARNING: terminal_path not found — terminal constraint DISABLED.")
except Exception as _te:
    vt_terminals_raw      = None
    vt_terminal_avail_raw = None
    print(f"  WARNING: Could not load terminal data ({_te}) — constraint DISABLED.")

# ── Load demand file (DAILY demand — used directly, no division) ──
demand_monthly_raw = {}   # part -> back-calculated monthly demand (for display only)
demand_daily_raw   = {}   # part -> daily demand (read directly from file)

try:
    demand_df = pd.read_excel(demand_path, sheet_name=DEMAND_SHEET_NAME)
    _part_col_d = next(
        (c for c in demand_df.columns if str(c).strip().lower() == DEMAND_PART_COL.lower()),
        None,
    )
    _dem_col_d = next(
        (c for c in demand_df.columns if str(c).strip().lower() == DEMAND_DAILY_COL.lower()),
        None,
    )
    if _part_col_d is None or _dem_col_d is None:
        print(f"  WARNING: Demand file missing '{DEMAND_PART_COL}' or "
              f"'{DEMAND_DAILY_COL}' column — demand constraint DISABLED.")
    else:
        for _, row in demand_df.iterrows():
            p = row[_part_col_d]
            v = row[_dem_col_d]
            if pd.isna(p) or str(p).strip() == "":
                continue
            part_key = str(p).strip()
            try:
                # v is DAILY demand — use directly
                daily_dem = float(v) if pd.notna(v) else 0.0
            except (ValueError, TypeError):
                daily_dem = 0.0
            # Store daily demand directly (NO division by WORKING_DAYS)
            demand_daily_raw[part_key]   = round(daily_dem, 4)
            # Back-calculate monthly for display purposes only
            demand_monthly_raw[part_key] = round(daily_dem * WORKING_DAYS, 4)
        print(f"  Demand data loaded (DAILY) : {len(demand_daily_raw)} parts from '{demand_path}'")
        demand_only_parts = sum(1 for p, d in demand_daily_raw.items() if d > 0)
        print(f"  Parts with positive demand : {demand_only_parts}")
except FileNotFoundError:
    print(f"  WARNING: demand_path '{demand_path}' not found — demand constraint DISABLED.")
except Exception as _de:
    print(f"  WARNING: Could not load demand data ({_de}) — demand constraint DISABLED.")

# =============================================================
# SECTION 5A — EFFECTIVE DAILY / MONTHLY HELPERS
# =============================================================

def effective_daily(part: str) -> float:
    """Return max(indent_daily, demand_daily) for a part.

    Rules
    -----
    * If demand_daily exists and > indent_daily → use demand (demand drives)
    * If indent_daily exists and >= demand_daily → use indent (normal flow)
    * A part with zero indent but positive demand still enters the scheduler
      (demand_only parts) — checked separately in skip logic.
    """
    ind = indent_daily.get(part, 0.0)
    dem = demand_daily_raw.get(part, 0.0)
    return max(ind, dem)

def effective_monthly(part: str) -> float:
    """Return max(indent_monthly, demand_monthly) for display."""
    ind = indent_monthly.get(part, 0.0)
    dem = demand_monthly_raw.get(part, 0.0)
    return max(ind, dem)

def demand_driver(part: str) -> str:
    """Return a short tag explaining which value is driving production."""
    ind = indent_daily.get(part, 0.0)
    dem = demand_daily_raw.get(part, 0.0)
    if dem > ind + 0.001:
        return f"DEMAND({dem:.2f}>indent {ind:.2f})"
    elif ind > dem + 0.001:
        return f"INDENT({ind:.2f})"
    elif ind > 0:
        return f"EQUAL({ind:.2f})"
    else:
        return "NO_DRIVER"

# =============================================================
# SECTION 6 — PARSE VT SHEET
# =============================================================

def find_col(df, name, sheet):
    match = next(
        (c for c in df.columns if str(c).strip().lower() == name.lower()), None
    )
    if match is None:
        raise ValueError(
            f"Column '{name}' not found in sheet '{sheet}'.\n"
            f"Available: {list(df.columns)}"
        )
    return match

vt_col_part      = find_col(vt_parts_raw, "Part",       "VT")
vt_col_cycletime = find_col(vt_parts_raw, "Cycle time", "VT")
vt_col_cavity    = find_col(vt_parts_raw, "Cavity",     "VT")
vt_col_inventory = find_col(vt_parts_raw, "Inventory",  "VT")
vt_col_indent    = find_col(vt_parts_raw, "Indent",     "VT")
vt_col_tools     = find_col(vt_parts_raw, "Tools",      "VT")
vt_col_color     = find_col(vt_parts_raw, "Color",      "VT")

data = vt_parts_raw[
    vt_parts_raw[vt_col_part].notna() &
    (vt_parts_raw[vt_col_part].astype(str).str.strip() != "")
].copy()
data = data.drop_duplicates(subset=vt_col_part).copy()
data["Material"] = data[vt_col_part].astype(str).str.strip()

data["_ct"] = pd.to_numeric(data[vt_col_cycletime], errors="coerce").replace(0, np.nan)
data["Rate"] = 3600 / data["_ct"]

data_valid     = data[data["Rate"].notna()].copy()
data_zero_rate = data[data["Rate"].isna()].copy()

print(f"  VT parts in sheet       : {len(data)}")
print(f"  Parts with valid rate   : {len(data_valid)}")

# =============================================================
# SECTION 7 — LOOKUP DICTIONARIES
# =============================================================

def safe_dict(df, key_col, val_col, default=0.0):
    return {
        str(k).strip(): (default if pd.isna(v) else float(v))
        for k, v in zip(df[key_col], df[val_col])
        if pd.notna(k) and str(k).strip() != ""
    }

inventory      = safe_dict(data,       "Material", vt_col_inventory)
rate           = safe_dict(data_valid, "Material", "Rate")
indent_monthly = safe_dict(data,       "Material", vt_col_indent)

# indent_daily stays as pure indent (do NOT replace with effective)
indent_daily = {
    p: round(qty / WORKING_DAYS, 4)
    for p, qty in indent_monthly.items()
}

tools_available = {}
part_color      = {}

for _, row in data.iterrows():
    p = str(row["Material"]).strip()
    v = row[vt_col_tools]
    tools_available[p] = (
        max(1, int(float(v)))
        if pd.notna(v) and str(v).strip() != ""
        else 1
    )
    c = row[vt_col_color]
    part_color[p] = (
        str(c).strip().upper()
        if pd.notna(c) and str(c).strip() not in ("", "nan")
        else "UNKNOWN"
    )

ALL_KNOWN_COLORS = dict(part_color)

color_groups = {}
for p, c in part_color.items():
    color_groups.setdefault(c, []).append(p)
print(f"  Distinct colours        : {len(color_groups)}")
for col, pts in sorted(color_groups.items()):
    print(f"    {col:<20} -> {len(pts)} part(s)")

# today_target_qty uses effective_daily so demand is reflected
today_target_qty = {
    p: max(0.0, effective_daily(p) - inventory.get(p, 0.0))
    for p in set(list(indent_monthly.keys()) + list(demand_daily_raw.keys()))
}

# =============================================================
# SECTION 7A — FIXED MACHINE CONSTRAINT
# =============================================================

def build_fixed_machine_dicts(df):
    pfm = {}
    mfp = {}
    if df is None or df.empty:
        return pfm, mfp
    machine_col = next(
        (c for c in df.columns if str(c).strip().lower() == "machine"), None
    )
    if machine_col is None:
        print("  WARNING: VT_Fixed has no 'Machine' column — fixed constraint disabled")
        return pfm, mfp
    part_cols = [c for c in df.columns if str(c).strip().lower() != "machine"]
    if not part_cols:
        print("  WARNING: VT_Fixed has no part columns — fixed constraint disabled")
        return pfm, mfp
    for _, row in df.iterrows():
        machine = row[machine_col]
        if pd.isna(machine) or str(machine).strip() == "":
            continue
        m = str(machine).strip()
        for col in part_cols:
            val = row[col]
            if pd.isna(val) or str(val).strip() in ("", "nan"):
                continue
            p = str(val).strip()
            if p in pfm:
                print(f"  WARNING: Part '{p}' in VT_Fixed more than once — keeping {pfm[p]}")
                continue
            pfm[p] = m
            mfp.setdefault(m, []).append(p)
    return pfm, mfp

part_fixed_machine, machine_fixed_parts = build_fixed_machine_dicts(vt_fixed_raw)

print(f"  Fixed machine mappings  : {len(part_fixed_machine)} parts")
if part_fixed_machine:
    for m, parts in sorted(machine_fixed_parts.items()):
        print(f"    {m:<25} <- {', '.join(parts)}")

# =============================================================
# SECTION 7A2 — FIXED MACHINE PHASE HELPERS
# =============================================================

def fixed_machine_phase(machine, current_inventory):
    fixed_parts = machine_fixed_parts.get(machine, [])
    if not fixed_parts:
        return "B"
    for p in fixed_parts:
        daily = effective_daily(p)
        inv   = current_inventory.get(p, 0)
        if daily > 0 and inv < SAFETY_DAYS * daily:
            return "A"
    return "B"

def pick_fixed_part_for_today(machine, current_inventory, machine_last_part):
    fixed_parts = machine_fixed_parts.get(machine, [])
    candidates  = []
    for p in fixed_parts:
        daily = effective_daily(p)
        r_val = rate.get(p, 1)
        inv   = current_inventory.get(p, 0)
        if daily <= 0:
            continue
        days_cov     = inv / daily
        hours_needed = daily / r_val if r_val > 0 else 0
        candidates.append((p, days_cov, hours_needed))
    if not candidates:
        return None
    candidates.sort(key=lambda x: (round(x[1], 4), -x[2]))
    if len(candidates) >= 2:
        top, second = candidates[0], candidates[1]
        if abs(top[1] - second[1]) < 0.01:
            last_ran = machine_last_part.get(machine)
            if last_ran == top[0]:
                return second[0]
    return candidates[0][0]

def fixed_part_run_hours(part, phase):
    daily = effective_daily(part)
    r_val = rate.get(part, 1)
    if daily <= 0 or r_val <= 0:
        return AVAILABLE_HOURS
    indent_hrs = daily / r_val
    if phase == "A":
        return AVAILABLE_HOURS_EXTENDED if indent_hrs > 20.0 else AVAILABLE_HOURS
    else:
        return max(MIN_RUN_HOURS, indent_hrs)

# =============================================================
# SECTION 7B — TERMINAL CONSTRAINT
# =============================================================

def _build_part_terminals(df):
    result = {}
    if df is None or df.empty:
        return result
    part_col = next(
        (c for c in df.columns if str(c).strip().lower() in ("part", "material")), None
    )
    if part_col is None:
        print("  WARNING: VT_Terminals has no 'Part'/'Material' column")
        return result
    terminal_cols = [
        c for c in df.columns
        if str(c).strip().lower() not in ("part", "material")
    ]
    for _, row in df.iterrows():
        part = row[part_col]
        if pd.isna(part) or str(part).strip() == "":
            continue
        p = str(part).strip()
        terminals = []
        for col in terminal_cols:
            val = row[col]
            if pd.notna(val) and str(val).strip() not in ("", "nan"):
                terminals.append(str(val).strip().upper())
        if terminals:
            result[p] = terminals
    return result

def _build_terminal_status(df):
    result = {}
    if df is None or df.empty:
        return result
    term_col = next(
        (c for c in df.columns if str(c).strip().lower() == "terminal"), None
    )
    inv_col = next(
        (c for c in df.columns if str(c).strip().lower() == "inventory"), None
    )
    if term_col is None or inv_col is None:
        print("  WARNING: VT_Terminal_Inventory missing 'Terminal' or 'Inventory' column")
        return result
    for _, row in df.iterrows():
        t = row[term_col]
        v = row[inv_col]
        if pd.isna(t) or str(t).strip() == "":
            continue
        key = str(t).strip().upper()
        try:
            qty = float(v) if pd.notna(v) else 0.0
        except (ValueError, TypeError):
            qty = 0.0
        result[key] = qty
    return result

part_terminals  = _build_part_terminals(vt_terminals_raw)
terminal_status = _build_terminal_status(vt_terminal_avail_raw)

if terminal_status:
    zero_count  = sum(1 for v in terminal_status.values() if v <= 0)
    avail_count = len(terminal_status) - zero_count
    print(f"  Terminals loaded : {len(terminal_status)} total  |  "
          f"{avail_count} with stock  |  {zero_count} at ZERO inventory")
else:
    print(f"  Terminals        : no data loaded — constraint inactive")

def terminal_blocked(part, category_override=None):
    required = part_terminals.get(part, [])
    if not required:
        return False, ""
    cat   = category_override or part_category.get(part, "Stranger")
    daily = effective_daily(part)
    mult  = TERMINAL_THRESHOLD.get(cat, 0.25)
    threshold = mult * daily
    blocking = []
    for t in required:
        t_inv = terminal_status.get(t, 0)
        if t_inv < threshold:
            blocking.append(
                f"{t}(inv={t_inv:.0f} < need={threshold:.0f} [{cat} x{mult}])"
            )
    if blocking:
        return True, (
            f"Terminal inadequate: {', '.join(blocking)}  "
            f"(requires: {', '.join(required)})"
        )
    return False, ""

# =============================================================
# SECTION 7C — SKIP RULES  (DEMAND-AWARE)
# =============================================================

def should_skip(part):
    """
    Updated skip logic:
    - A part MUST NOT be skipped if it has positive demand, even if indent
      is zero or below the MIN_DAILY_INDENT threshold.
    - Inventory ceiling check uses effective_daily.
    - Terminal block is still a hard skip.
    """
    ind_daily  = indent_daily.get(part, 0.0)
    dem_daily  = demand_daily_raw.get(part, 0.0)
    eff_daily  = effective_daily(part)
    monthly    = indent_monthly.get(part, 0.0)
    r          = rate.get(part, 1.0)

    # If demand is positive, never skip for low-indent reasons
    has_demand = dem_daily > 0.0

    if not has_demand:
        if ind_daily <= MIN_DAILY_INDENT:
            return True, f"Daily indent {ind_daily:.2f} <= {MIN_DAILY_INDENT} threshold (no demand)"

        indent_hrs = monthly / r if r > 0 else 0.0
        if indent_hrs <= MIN_INDENT_HOURS:
            return True, f"Whole monthly indent = {indent_hrs:.2f}h <= {MIN_INDENT_HOURS}h threshold (no demand)"

    inv = inventory.get(part, 0.0)
    if eff_daily > 0 and inv >= TARGET_DAYS * eff_daily:
        return True, (
            f"Inventory ({inv:.0f}) >= {TARGET_DAYS}-day target "
            f"({TARGET_DAYS * eff_daily:.0f} pcs using eff_daily={eff_daily:.2f}) — at ceiling"
        )

    t_blocked, t_reason = terminal_blocked(part)
    if t_blocked:
        return True, t_reason

    return False, ""

def is_hard_skip(part):
    if terminal_blocked(part)[0]:
        return True
    return False

def is_scheduling_skip(part):
    return should_skip(part)[0]

# =============================================================
# SECTION 7D — CHANGEOVER TIMES
# =============================================================

def build_changeover_dict(co_df):
    co_dict = {}
    for _, row in co_df.iterrows():
        machine = str(row["Unique Machines"]).strip()
        minutes = row["Changeover time"]
        if pd.notna(minutes) and machine:
            co_dict[machine] = float(minutes) / 60.0
    return co_dict

vt_changeover          = build_changeover_dict(vt_co_raw)
DEFAULT_CHANGEOVER_HRS = 40 / 60.0

# =============================================================
# SECTION 7E — MACHINE PART COUNT
# =============================================================

def build_machine_part_count(df):
    mpc = {}
    machine_col = next(
        (c for c in df.columns if str(c).strip().lower() == "machine"), None
    )
    count_col = next(
        (c for c in df.columns if str(c).strip().lower() == "part_count"), None
    )
    if machine_col is None or count_col is None:
        print("  WARNING: VT_Machine_Part_Count missing columns")
        return {}
    for _, row in df.iterrows():
        m = str(row[machine_col]).strip()
        v = row[count_col]
        if m and pd.notna(v):
            try:
                mpc[m] = int(float(v))
            except (ValueError, TypeError):
                pass
    return mpc

machine_part_count = build_machine_part_count(vt_machine_count_raw)
max_part_count     = max(machine_part_count.values(), default=1) or 1
print(f"  Machine part counts loaded: {len(machine_part_count)} machines")

# =============================================================
# SECTION 7F — PART CATEGORY
# =============================================================

def build_category(df):
    cat      = {}
    part_col = next(
        (c for c in df.columns if str(c).strip().lower() == "part"), None
    )
    cat_col  = next(
        (c for c in df.columns if str(c).strip().lower() == "category"), None
    )
    if part_col is None:
        return cat
    for _, row in df.iterrows():
        part = row[part_col]
        if pd.isna(part) or str(part).strip() == "":
            continue
        val = "Stranger"
        if cat_col and pd.notna(row[cat_col]):
            val = str(row[cat_col]).strip().capitalize()
            if val not in ("Runner", "Repeater", "Stranger"):
                val = "Stranger"
        cat[str(part).strip()] = val
    return cat

part_category  = build_category(vt_parts_raw)
CATEGORY_SCORE = {"Runner": 100, "Repeater": 60, "Stranger": 20}

# Parts that appear only in demand (not in Book1) get category Stranger by default
for p in demand_daily_raw:
    if p not in part_category:
        part_category[p] = "Stranger"

# =============================================================
# SECTION 8 — MACHINE STATE
# =============================================================

def load_machine_state():
    if os.path.exists(MACHINE_STATE_FILE):
        try:
            with open(MACHINE_STATE_FILE) as f:
                content = f.read().strip()
            if not content:
                print(f"  Machine state : file empty — first run")
                os.remove(MACHINE_STATE_FILE)
                return {}
            state = json.loads(content)
            if not isinstance(state, dict):
                print(f"  Machine state : corrupt — first run")
                os.remove(MACHINE_STATE_FILE)
                return {}
            unknown = [p for p in state.values() if p not in part_color]
            if unknown:
                print(f"  Machine state : {len(unknown)} part(s) not in today's sheet:")
                for p in unknown:
                    print(f"    {p} — kept for CO calculation")
                    ALL_KNOWN_COLORS[p] = "NEEDS_PURGE"
            print(f"  Machine state loaded  ({len(state)} machines)")
            return state
        except Exception as e:
            print(f"  Machine state : error ({e}) — first run")
            os.remove(MACHINE_STATE_FILE)
            return {}
    print(f"  Machine state : FIRST RUN — no changeover today")
    return {}

def save_machine_state(state):
    combined = {m: p for m, p in state.items() if p is not None}
    with open(MACHINE_STATE_FILE, "w") as f:
        json.dump(combined, f, indent=2)
    print(f"\n  Machine state saved -> '{MACHINE_STATE_FILE}'")

machine_state = load_machine_state()

# =============================================================
# SECTION 9 — COMPATIBILITY MATRIX
# =============================================================

def build_compatibility(matrix):
    machines = matrix.columns[1:].tolist()
    compat   = {}
    for _, row in matrix.iterrows():
        part = row["Part"]
        for m in machines:
            if row[m] == 1:
                compat.setdefault(part, []).append(m)
    return compat, machines

vt_compat, vt_machines = build_compatibility(vt_matrix)

machine_compatible_parts = {m: [] for m in vt_machines}
for p, machines in vt_compat.items():
    for m in machines:
        if m in machine_compatible_parts:
            machine_compatible_parts[m].append(p)

# =============================================================
# SECTION 10 — SCENARIO CLASSIFIER  (DEMAND-AWARE)
# =============================================================

def classify_scenario(parts):
    coverage = []
    for p in parts:
        inv   = inventory.get(p, 0)
        daily = effective_daily(p)
        skip, _ = should_skip(p)
        if skip or daily == 0:
            continue
        coverage.append(inv / daily)
    if not coverage:
        return 3, "SCENARIO 3 — No active parts today"
    n        = len(coverage)
    critical = sum(1 for c in coverage if c < 1)
    low      = sum(1 for c in coverage if c < SAFETY_DAYS)
    if critical == n:
        return 0, f"SCENARIO 0 — ALL {n} active parts critical (inv < 1 day)"
    elif critical > 0:
        return 1, f"SCENARIO 1 — {critical}/{n} parts critical  |  {n-critical} have buffer"
    elif low > 0:
        return 2, f"SCENARIO 2 — {low}/{n} parts below {SAFETY_DAYS}-day safety floor"
    else:
        return 3, f"SCENARIO 3 — All {n} parts healthy (>={SAFETY_DAYS} days safety floor)"

# =============================================================
# SECTION 11 — OPD CAP  (DEMAND-AWARE)
# =============================================================

def opd_cap(scenario_id):
    scenario_opd = {
        0: OPD_SCENARIO_0,
        1: OPD_SCENARIO_1,
        2: OPD_SCENARIO_2,
        3: OPD_SCENARIO_3,
    }.get(scenario_id, OPD_SCENARIO_2)
    return min(scenario_opd, TARGET_DAYS)

def effective_opd_cap_qty(part, scenario_id, current_inv):
    """
    Return the effective production ceiling quantity for a part.

    For demand-driven parts the OPD cap is computed on effective_daily,
    but we also guarantee at least 1 day of demand coverage above current
    inventory so today's demand is always met.
    """
    eff_d  = effective_daily(part)
    dem_d  = demand_daily_raw.get(part, 0.0)
    cap_q  = opd_cap(scenario_id) * eff_d

    # Demand floor: we must always be able to cover at least today's demand
    demand_floor_qty = max(0.0, dem_d - current_inv) if dem_d > 0 else 0.0

    return max(cap_q, demand_floor_qty)

# =============================================================
# SECTION 12 — PRIORITY SCORING  (DEMAND-AWARE)
# =============================================================

def compute_priority_scores(active_parts):
    rows = []
    for p in active_parts:
        inv      = inventory.get(p, 0)
        daily    = effective_daily(p)
        cat      = part_category.get(p, "Stranger")
        days_cov = inv / daily if daily > 0 else 999.0
        gap_score = min(1.0, max(0.0, (TARGET_DAYS - days_cov) / TARGET_DAYS))
        velocity_raw = daily / max(float(inv), 1.0) if daily > 0 else 0.0
        rows.append({
            "part": p, "inv": inv, "daily": daily,
            "days_cov": days_cov, "cat": cat,
            "gap_score": gap_score, "velocity_raw": velocity_raw,
        })
    if not rows:
        return {}, []
    max_daily    = max(r["daily"]        for r in rows) or 1.0
    max_velocity = max(r["velocity_raw"] for r in rows) or 1.0
    scores, score_rows = {}, []
    for r in rows:
        p = r["part"]
        gap_pct      = r["gap_score"] * 100.0
        velocity_pct = (r["velocity_raw"] / max_velocity) * 100.0
        urgency_score  = 0.60 * gap_pct + 0.40 * velocity_pct
        category_score = CATEGORY_SCORE.get(r["cat"], 20)
        indent_score   = (r["daily"] / max_daily) * 100.0
        final_score = (
            W_URGENCY  * urgency_score +
            W_CATEGORY * category_score +
            W_INDENT   * indent_score
        )
        scores[p] = round(final_score, 2)
        score_rows.append({
            "Part": p, "Category": r["cat"],
            "Color": part_color.get(p, "UNKNOWN"),
            "Fixed_Machine": part_fixed_machine.get(p, "—"),
            "Tools": tools_available.get(p, 1),
            "Inventory_Now": round(r["inv"], 0),
            "Indent_Daily": round(indent_daily.get(p, 0.0), 2),
            "Demand_Daily": round(demand_daily_raw.get(p, 0.0), 2),
            "Effective_Daily": round(r["daily"], 2),
            "Demand_Driver": demand_driver(p),
            "Days_Coverage": round(r["days_cov"], 2),
            "Safety_Floor": SAFETY_DAYS, "Target_Ceiling": TARGET_DAYS,
            "Buffer_Status": (
                "CRITICAL"     if r["days_cov"] < 1 else
                "BELOW_SAFETY" if r["days_cov"] < SAFETY_DAYS else
                "BUILDING"     if r["days_cov"] < TARGET_DAYS else
                "AT_TARGET"
            ),
            "Gap_Score_Pct": round(gap_pct, 1),
            "Velocity_Score_Pct": round(velocity_pct, 1),
            "Urgency_Score": round(urgency_score, 1),
            "Category_Score": category_score,
            "Indent_Score": round(indent_score, 1),
            "Final_Score": round(final_score, 2),
        })
    return scores, score_rows

# =============================================================
# SECTION 13 — COLOUR-AWARE CHANGEOVER HELPER
# =============================================================

def _co_hrs_for(part, machine, machine_last_part):
    last = machine_last_part.get(machine)
    if last is None or last == part:
        return 0.0
    base_co    = vt_changeover.get(machine, DEFAULT_CHANGEOVER_HRS)
    last_color = ALL_KNOWN_COLORS.get(last, "UNKNOWN")
    new_color  = part_color.get(part, "UNKNOWN")
    if last_color == "NEEDS_PURGE":
        return base_co + COLOR_PURGE_HRS
    purge = (
        COLOR_PURGE_HRS
        if last_color != new_color
        and last_color not in ("UNKNOWN",)
        and new_color  not in ("UNKNOWN",)
        else 0.0
    )
    return base_co + purge

# =============================================================
# SECTION 14 — MACHINE RANKER
# =============================================================

def rank_machines(part, machines_to_try, machine_hours,
                  machine_last_part, inv_days,
                  exclude_fixed_machines=True,
                  allow_fixed_overflow=False):
    category  = part_category.get(part, "Stranger")
    new_color = part_color.get(part, "UNKNOWN")
    fixed_m   = part_fixed_machine.get(part)
    is_fixed  = fixed_m is not None
    if is_fixed:
        runner_lock = False
    else:
        runner_lock = (category == "Runner" and inv_days < RUNNER_PRIORITY_DAYS)

    effective_candidates = []
    for m in machines_to_try:
        if exclude_fixed_machines and not allow_fixed_overflow and m in machine_fixed_parts:
            if not is_fixed:
                continue
            elif m != fixed_m:
                continue
        effective_candidates.append(m)

    if is_fixed and fixed_m in effective_candidates:
        used_f = machine_hours.get(fixed_m, 0)
        free_f = round(AVAILABLE_HOURS - used_f, 4)
        co_f   = _co_hrs_for(part, fixed_m, machine_last_part)
        eff_f  = round(free_f - co_f, 4)
        if eff_f >= MIN_RUN_HOURS:
            fallback = _rank_normal(
                part,
                [m for m in effective_candidates if m != fixed_m],
                machine_hours, machine_last_part, new_color, runner_lock
            )
            return [(fixed_m, co_f, eff_f, -1.0)] + fallback, runner_lock
        remaining = [m for m in effective_candidates if m != fixed_m]
        return _rank_normal(
            part, remaining, machine_hours, machine_last_part,
            new_color, runner_lock
        ), runner_lock

    return _rank_normal(
        part, effective_candidates, machine_hours, machine_last_part,
        new_color, runner_lock
    ), runner_lock

def _rank_normal(part, machines_to_try, machine_hours,
                 machine_last_part, new_color, runner_lock):
    ranked = []
    for m in machines_to_try:
        used = machine_hours.get(m, 0)
        free = round(AVAILABLE_HOURS - used, 4)
        if free < MIN_RUN_HOURS:
            continue
        last = machine_last_part.get(m)
        if runner_lock and last != part:
            continue
        if last is None or last == part:
            co_hrs      = 0.0
            color_bonus = 0.0
        else:
            base_co    = vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS)
            last_color = ALL_KNOWN_COLORS.get(last, "UNKNOWN")
            same_color = (
                last_color == new_color
                and last_color not in ("UNKNOWN", "NEEDS_PURGE")
                and new_color  not in ("UNKNOWN",)
            )
            purge = (
                0.0 if same_color else (
                    COLOR_PURGE_HRS
                    if last_color not in ("UNKNOWN",) and new_color not in ("UNKNOWN",)
                    else 0.0
                )
            )
            co_hrs      = base_co + purge
            color_bonus = -0.08 if same_color else 0.0
        effective_free = round(free - co_hrs, 4)
        if effective_free < MIN_RUN_HOURS:
            continue
        part_count      = machine_part_count.get(m, max_part_count)
        count_score     = part_count / max_part_count
        co_penalty      = (co_hrs / AVAILABLE_HOURS) * 0.3
        util_penalty    = (used  / AVAILABLE_HOURS) * 0.2
        same_part_bonus = -0.15 if (last == part) else 0.0
        cost            = (count_score + co_penalty + util_penalty
                           + same_part_bonus + color_bonus)
        ranked.append((m, co_hrs, effective_free, cost))
    ranked.sort(key=lambda x: x[3])
    return ranked

_phase_a_machines: set = set()

# =============================================================
# SECTION 14A — PLANNED QTY TRACKING HELPERS
# =============================================================

def _get_part_total_qty(part, plan):
    return sum(float(r.get("Production_Qty", 0)) for r in plan if r["Part"] == part)

def _get_part_machines(part, plan):
    return {r["Machine"] for r in plan if r["Part"] == part}

def _is_indent_met(part, plan):
    """
    Indent/demand is considered met when total production + inventory
    covers today's effective_daily target.
    """
    daily = effective_daily(part)
    if daily <= 0:
        return True
    total_qty = _get_part_total_qty(part, plan)
    inv_now   = inventory.get(part, 0)
    return (inv_now + total_qty) >= (daily - 0.5)

def _parts_with_indent_met(plan):
    from collections import defaultdict
    part_qty = defaultdict(float)
    for row in plan:
        part_qty[row["Part"]] += float(row.get("Production_Qty", 0))
    result = set()
    for p, qty in part_qty.items():
        daily = effective_daily(p)
        inv_now = inventory.get(p, 0)
        if daily > 0 and (inv_now + qty) >= (daily - 0.5):
            result.add(p)
    return result

def _current_co_count(plan):
    return sum(1 for r in plan if r.get("Changeover") == "Yes")

# Helper: build a standard plan row dict
def _make_row(part, machine, run_hrs, co_hrs, qty, scenario_id,
              type_label, role_label, runner_lock=False,
              phase=1, stagger="No"):
    daily    = effective_daily(part)
    monthly  = effective_monthly(part)
    r_val    = rate.get(part, 1)
    inv_now  = inventory.get(part, 0)
    color    = part_color.get(part, "UNKNOWN")
    fixed_m  = part_fixed_machine.get(part)
    last     = machine_state.get(machine)
    l_color  = ALL_KNOWN_COLORS.get(last, "UNKNOWN") if last else "UNKNOWN"
    has_purge = (last is not None and last != part and color != l_color
                 and color != "UNKNOWN" and l_color not in ("UNKNOWN", "NEEDS_PURGE"))
    fixed_used = "YES" if (fixed_m and machine == fixed_m) else (
                 "FALLBACK" if fixed_m else "N/A")
    indent_met_flag = (
        "YES" if (inv_now + qty) >= (daily - 0.5)
        else f"NO — need {daily:.2f}/d, have {inv_now+qty:.0f} pcs"
    )
    return {
        "Part": part,
        "Color": color,
        "Category": part_category.get(part, "Stranger"),
        "Fixed_Machine": fixed_m or "—",
        "Fixed_Used": fixed_used,
        "Machine": machine,
        "Run_Hours": round(run_hrs, 3),
        "Changeover_Hrs": round(co_hrs, 3),
        "Total_Hrs_Used": round(co_hrs + run_hrs, 3),
        "Rate_Per_Hour": round(r_val, 2),
        "Production_Qty": qty,
        "Indent_Daily": round(indent_daily.get(part, 0.0), 2),
        "Demand_Daily": round(demand_daily_raw.get(part, 0.0), 2),
        "Effective_Daily": round(daily, 2),
        "Demand_Driver": demand_driver(part),
        "Monthly_Indent": round(monthly, 0),
        "Daily_Indent": round(daily, 2),
        "Today_Target": round(today_target_qty.get(part, 0), 0),
        "Changeover": "No" if co_hrs == 0 else "Yes",
        "Color_Purge": "Yes" if has_purge else "No",
        "Type": type_label,
        "Role": role_label,
        "Tools_Available": tools_available.get(part, 1),
        "Tools_Used": 1,
        "Runner_Lock": "YES" if runner_lock else "No",
        "Priority_Score": 0,
        "Phase": phase,
        "Indent_Met": indent_met_flag,
        "Stagger_Adjusted": stagger,
    }

# =============================================================
# SECTION 14B — INTRA-MACHINE CO RESEQUENCING
# =============================================================

def resequence_machine_rows(plan, machine_last_part_yesterday):
    from collections import defaultdict
    machine_rows = defaultdict(list)
    other_rows   = []
    for row in plan:
        m = row.get("Machine")
        if m in vt_machines:
            machine_rows[m].append(row)
        else:
            other_rows.append(row)
    resequenced_plan = []
    for m in vt_machines:
        rows = machine_rows.get(m, [])
        if len(rows) <= 1:
            resequenced_plan.extend(rows)
            continue
        yesterday_part = machine_last_part_yesterday.get(m)
        ordered   = []
        remaining = list(rows)
        seed = None
        if yesterday_part:
            for r in remaining:
                if r["Part"] == yesterday_part:
                    seed = r
                    break
        if seed is None:
            seed = max(remaining, key=lambda r: float(r.get("Priority_Score", 0) or 0))
        ordered.append(seed)
        remaining.remove(seed)
        while remaining:
            last_part  = ordered[-1]["Part"]
            last_color_val = part_color.get(last_part, "UNKNOWN")
            def _co_cost(r, _lc=last_color_val):
                p = r["Part"]
                c = part_color.get(p, "UNKNOWN")
                if p == last_part:
                    return -1.0
                base = vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS)
                purge = (
                    COLOR_PURGE_HRS
                    if _lc not in ("UNKNOWN", "NEEDS_PURGE")
                    and c not in ("UNKNOWN",)
                    and _lc != c
                    else 0.0
                )
                return base + purge
            remaining.sort(key=_co_cost)
            ordered.append(remaining.pop(0))
        for i, row in enumerate(ordered):
            p = row["Part"]
            prev_part = yesterday_part if i == 0 else ordered[i - 1]["Part"]
            if prev_part is None or prev_part == p:
                new_co = 0.0
            else:
                base_co    = vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS)
                prev_color = ALL_KNOWN_COLORS.get(prev_part, "UNKNOWN")
                new_color  = part_color.get(p, "UNKNOWN")
                purge = (
                    COLOR_PURGE_HRS
                    if prev_color not in ("UNKNOWN", "NEEDS_PURGE")
                    and new_color not in ("UNKNOWN",)
                    and prev_color != new_color
                    else 0.0
                )
                new_co = base_co + purge
            row["Changeover_Hrs"]  = round(new_co, 3)
            row["Changeover"]      = "No" if new_co == 0 else "Yes"
            row["Total_Hrs_Used"]  = round(new_co + float(row.get("Run_Hours", 0) or 0), 3)
            row["Color_Purge"] = "Yes" if (
                new_co > vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS) + 0.001
            ) else "No"
        resequenced_plan.extend(ordered)
    resequenced_plan.extend(other_rows)
    return resequenced_plan

# =============================================================
# SECTION 14C — MACHINE HOURS RECONCILER
# =============================================================

def reconcile_machine_hours(plan, machine_hours):
    recomputed = {m: 0.0 for m in vt_machines}
    for row in plan:
        m = row.get("Machine")
        if m in recomputed:
            recomputed[m] += float(row.get("Run_Hours", 0) or 0) + float(row.get("Changeover_Hrs", 0) or 0)
    for m in vt_machines:
        machine_hours[m] = round(recomputed[m], 4)

# =============================================================
# SECTION 15 — FIXED MACHINE SCHEDULING PASS  (DEMAND-AWARE)
# =============================================================

def schedule_fixed_machines(machine_hours, machine_last_part,
                              current_inventory, plan, already_planned,
                              priority_scores, scenario_id):
    print(f"\n{'─'*65}")
    print(f"  FIXED MACHINE SCHEDULING PASS")
    print(f"{'─'*65}")

    phase_a_machines = set()
    fixed_plan_rows  = []

    for machine, fixed_parts in sorted(machine_fixed_parts.items()):
        phase = fixed_machine_phase(machine, current_inventory)
        inv_summary = []
        for p in fixed_parts:
            daily    = effective_daily(p)
            inv      = current_inventory.get(p, 0)
            days_cov = inv / daily if daily > 0 else 999
            r_val    = rate.get(p, 1)
            hrs_need = daily / r_val if r_val > 0 else 0
            inv_summary.append(f"{p}(inv={inv:.0f}={days_cov:.2f}d, need={hrs_need:.1f}h/d, driver={demand_driver(p)})")
        print(f"\n  Machine: {machine}  Phase: {phase}")
        print(f"    Fixed parts: {' | '.join(inv_summary)}")

        if phase == "A":
            phase_a_machines.add(machine)
            chosen = pick_fixed_part_for_today(machine, current_inventory, machine_last_part)
            if chosen is None:
                print(f"    No eligible fixed part — machine skipped")
                continue
            daily   = effective_daily(chosen)
            r_val   = rate.get(chosen, 1)
            monthly = effective_monthly(chosen)
            color   = part_color.get(chosen, "UNKNOWN")
            score   = priority_scores.get(chosen, 0)
            run_hrs_cap = fixed_part_run_hours(chosen, "A")
            effective_run = run_hrs_cap
            qty = round(effective_run * r_val, 0)
            machine_hours[machine]      = round(effective_run, 4)
            current_inventory[chosen]   = round(current_inventory.get(chosen, 0) + qty, 0)
            machine_last_part[machine]  = chosen
            already_planned.add(chosen)
            row = {
                "Part": chosen, "Color": color,
                "Category": part_category.get(chosen, "Runner"),
                "Fixed_Machine": machine,
                "Fixed_Used": "YES — Phase A (buffer building)",
                "Machine": machine,
                "Run_Hours": round(effective_run, 3),
                "Changeover_Hrs": 0.0,
                "Total_Hrs_Used": round(effective_run, 3),
                "Rate_Per_Hour": round(r_val, 2),
                "Production_Qty": qty,
                "Indent_Daily": round(indent_daily.get(chosen, 0.0), 2),
                "Demand_Daily": round(demand_daily_raw.get(chosen, 0.0), 2),
                "Effective_Daily": round(daily, 2),
                "Demand_Driver": demand_driver(chosen),
                "Monthly_Indent": round(monthly, 0),
                "Daily_Indent": round(daily, 2),
                "Today_Target": round(today_target_qty.get(chosen, 0), 0),
                "Changeover": "No", "Color_Purge": "No",
                "Type": f"Fixed-PhaseA [{run_hrs_cap}h cap]",
                "Role": "Primary",
                "Tools_Available": tools_available.get(chosen, 1),
                "Tools_Used": 1, "Runner_Lock": "No",
                "Priority_Score": score, "Phase": 1,
                "Indent_Met": (
                    "YES" if (current_inventory.get(chosen, 0)) >= daily
                    else f"NO — need {daily:.2f}/d"
                ),
                "Stagger_Adjusted": "No",
            }
            plan.append(row)
            fixed_plan_rows.append(row)
            inv_after  = current_inventory.get(chosen, 0)
            days_after = inv_after / daily if daily > 0 else 0
            print(f"    Phase A -> {chosen}  {effective_run:.2f}h  qty={qty:.0f}  inv_after={inv_after:.0f} ({days_after:.2f}d)  driver={demand_driver(chosen)}")

        else:
            for chosen in fixed_parts:
                if machine_hours.get(machine, 0) >= AVAILABLE_HOURS - 0.05:
                    already_planned.add(chosen)
                    continue
                daily   = effective_daily(chosen)
                r_val   = rate.get(chosen, 1)
                monthly = effective_monthly(chosen)
                color   = part_color.get(chosen, "UNKNOWN")
                score   = priority_scores.get(chosen, 0)
                min_run_for_indent = fixed_part_run_hours(chosen, "B")
                available_now = round(AVAILABLE_HOURS - machine_hours.get(machine, 0), 4)
                effective_run = min(min_run_for_indent, available_now)
                if effective_run < MIN_RUN_HOURS:
                    already_planned.add(chosen)
                    continue
                inv_now_chosen = current_inventory.get(chosen, 0)
                cap_qty  = effective_opd_cap_qty(chosen, scenario_id, inv_now_chosen)
                headroom = max(0.0, cap_qty - inv_now_chosen)
                if headroom > 0 and r_val > 0:
                    max_hrs_for_opd = headroom / r_val
                    effective_run = min(available_now, max(effective_run, max_hrs_for_opd))
                    effective_run = max(effective_run, MIN_RUN_HOURS)
                qty = round(effective_run * r_val, 0)
                machine_hours[machine] = round(machine_hours.get(machine, 0) + effective_run, 4)
                current_inventory[chosen] = round(current_inventory.get(chosen, 0) + qty, 0)
                machine_last_part[machine] = chosen
                already_planned.add(chosen)
                inv_after_c = current_inventory.get(chosen, 0)
                indent_met  = inv_after_c >= (daily - 0.5)
                row = {
                    "Part": chosen, "Color": color,
                    "Category": part_category.get(chosen, "Runner"),
                    "Fixed_Machine": machine,
                    "Fixed_Used": "YES — Phase B (OPD build)",
                    "Machine": machine,
                    "Run_Hours": round(effective_run, 3),
                    "Changeover_Hrs": 0.0,
                    "Total_Hrs_Used": round(effective_run, 3),
                    "Rate_Per_Hour": round(r_val, 2),
                    "Production_Qty": qty,
                    "Indent_Daily": round(indent_daily.get(chosen, 0.0), 2),
                    "Demand_Daily": round(demand_daily_raw.get(chosen, 0.0), 2),
                    "Effective_Daily": round(daily, 2),
                    "Demand_Driver": demand_driver(chosen),
                    "Monthly_Indent": round(monthly, 0),
                    "Daily_Indent": round(daily, 2),
                    "Today_Target": round(today_target_qty.get(chosen, 0), 0),
                    "Changeover": "No", "Color_Purge": "No",
                    "Type": "Fixed-PhaseB",
                    "Role": "Primary",
                    "Tools_Available": tools_available.get(chosen, 1),
                    "Tools_Used": 1, "Runner_Lock": "No",
                    "Priority_Score": score, "Phase": 1,
                    "Indent_Met": (
                        "YES" if indent_met
                        else f"NO — partial (need {daily:.2f}/d)"
                    ),
                    "Stagger_Adjusted": "No",
                }
                plan.append(row)
                fixed_plan_rows.append(row)
                remaining_hrs = round(AVAILABLE_HOURS - machine_hours.get(machine, 0), 4)
                print(f"    Phase B -> {chosen}  {effective_run:.2f}h  qty={qty:.0f}  remaining={remaining_hrs:.2f}h  driver={demand_driver(chosen)}")

    print(f"\n  Fixed machine pass: {len(phase_a_machines)} Phase A  |  "
          f"{len(machine_fixed_parts) - len(phase_a_machines)} Phase B")
    return phase_a_machines, fixed_plan_rows

# =============================================================
# SECTION 16 — TOOL-AWARE ASSIGNMENT  (DEMAND-AWARE)
# =============================================================

def assign_part(part, scenario_id, machine_hours, machine_last_part,
                current_inventory, plan, already_planned, priority_scores):
    daily      = effective_daily(part)
    monthly    = effective_monthly(part)
    r_val      = rate.get(part, 1)
    inv_now    = current_inventory.get(part, 0)
    category   = part_category.get(part, "Stranger")
    tools      = tools_available.get(part, 1)
    score      = priority_scores.get(part, 0)
    color      = part_color.get(part, "UNKNOWN")
    inv_days   = inv_now / daily if daily > 0 else 999
    compatible = vt_compat.get(part, [])
    fixed_m    = part_fixed_machine.get(part)

    if not compatible:
        return []

    total_shortfall = max(0.0, daily - inv_now)
    hrs_for_full    = max(MIN_RUN_HOURS,
                          total_shortfall / r_val if r_val > 0 else MIN_RUN_HOURS)

    new_rows        = []
    produced_so_far = 0.0
    tools_used      = 0
    used_machines   = set()

    already_on_fixed = (
        fixed_m is not None
        and any(r["Machine"] == fixed_m and r["Part"] == part for r in plan)
    )

    if already_on_fixed:
        qty_from_fixed = sum(
            float(r["Production_Qty"])
            for r in plan if r["Part"] == part and r["Machine"] == fixed_m
        )
        produced_so_far = qty_from_fixed
        tools_used = 1
        used_machines.add(fixed_m)
        indent_met_on_fixed = (inv_now + produced_so_far >= total_shortfall - 0.5)
        if indent_met_on_fixed:
            already_planned.add(part)
            return []
        else:
            if tools <= 1:
                already_planned.add(part)
                return []
    else:
        ranked, runner_lock = rank_machines(
            part, compatible, machine_hours, machine_last_part, inv_days
        )
        if not ranked:
            return []
        m1, co1, eff1, _ = ranked[0]
        run1 = min(eff1, hrs_for_full)
        run1 = max(run1, MIN_RUN_HOURS)
        qty1 = round(run1 * r_val, 0)
        fixed_used = (fixed_m is not None and m1 == fixed_m)
        machine_hours[m1] = round(machine_hours.get(m1, 0) + co1 + run1, 4)
        current_inventory[part] = round(current_inventory.get(part, 0) + qty1, 0)
        machine_last_part[m1] = part
        produced_so_far += qty1
        tools_used += 1
        used_machines.add(m1)
        already_planned.add(part)
        indent_met_on_primary = (inv_now + produced_so_far >= total_shortfall - 0.5)
        last_m1 = machine_state.get(m1)
        purge_applied = (
            last_m1 is not None and last_m1 != part
            and ALL_KNOWN_COLORS.get(last_m1, "UNKNOWN") != color
            and ALL_KNOWN_COLORS.get(last_m1, "UNKNOWN") not in ("UNKNOWN", "NEEDS_PURGE")
            and color != "UNKNOWN"
        )
        type_tag = "Primary"
        if inv_now == 0:
            type_tag += " [ZERO-INV]"
        if fixed_used:
            type_tag += " [FIXED-MACHINE]"
        elif fixed_m is not None:
            type_tag += " [FIXED-FALLBACK]"

        new_rows.append({
            "Part": part, "Color": color, "Category": category,
            "Fixed_Machine": fixed_m or "—",
            "Fixed_Used": "YES" if fixed_used else ("FALLBACK" if fixed_m else "N/A"),
            "Machine": m1, "Run_Hours": round(run1, 3),
            "Changeover_Hrs": round(co1, 3),
            "Total_Hrs_Used": round(co1 + run1, 3),
            "Rate_Per_Hour": round(r_val, 2), "Production_Qty": qty1,
            "Indent_Daily": round(indent_daily.get(part, 0.0), 2),
            "Demand_Daily": round(demand_daily_raw.get(part, 0.0), 2),
            "Effective_Daily": round(daily, 2),
            "Demand_Driver": demand_driver(part),
            "Monthly_Indent": round(monthly, 0), "Daily_Indent": round(daily, 2),
            "Today_Target": round(today_target_qty.get(part, 0), 0),
            "Changeover": "No" if co1 == 0 else "Yes",
            "Color_Purge": "Yes" if purge_applied else "No",
            "Type": type_tag, "Role": "Primary",
            "Tools_Available": tools, "Tools_Used": 1,
            "Runner_Lock": "YES" if runner_lock else "No",
            "Priority_Score": score, "Phase": 1,
            "Indent_Met": (
                "YES" if indent_met_on_primary
                else f"NO — shortfall remains (need {daily:.2f}/d)"
            ),
            "Stagger_Adjusted": "No",
        })
        if indent_met_on_primary:
            _do_inv_build(part, m1, scenario_id, machine_hours,
                          current_inventory, new_rows, r_val, daily)
            for row in new_rows:
                row["Tools_Used"] = tools_used
            return new_rows

    is_critical = (inv_now == 0)
    if category in ("Stranger", "Repeater"):
        tool_hard_cap = 1
    else:
        tool_hard_cap = tools if is_critical else min(2, tools)

    if tool_hard_cap <= 1 and tools_used >= 1:
        if new_rows:
            _do_inv_build(part, new_rows[0]["Machine"], scenario_id, machine_hours,
                          current_inventory, new_rows, r_val, daily)
        for row in new_rows:
            row["Tools_Used"] = tools_used
        return new_rows

    while (produced_so_far < (total_shortfall - 0.5) and tools_used < tool_hard_cap):
        shortfall_now = total_shortfall - produced_so_far
        hrs_needed = max(MIN_RUN_HOURS, shortfall_now / r_val if r_val > 0 else MIN_RUN_HOURS)
        remaining_machines = [
            m for m in compatible
            if m not in used_machines and m not in machine_fixed_parts
        ]
        ranked_next, _ = rank_machines(
            part, remaining_machines, machine_hours, machine_last_part, inv_days
        )
        if not ranked_next:
            break
        mx, cox, effx, _ = ranked_next[0]
        run_x = max(MIN_RUN_HOURS, min(effx, hrs_needed))
        qty_x = round(run_x * r_val, 0)
        machine_hours[mx] = round(machine_hours.get(mx, 0) + cox + run_x, 4)
        current_inventory[part] = round(current_inventory.get(part, 0) + qty_x, 0)
        machine_last_part[mx] = part
        produced_so_far += qty_x
        tools_used += 1
        used_machines.add(mx)
        indent_met_here = (inv_now + produced_so_far >= total_shortfall - 0.5)
        last_mx = machine_state.get(mx)
        purge_x = (
            last_mx is not None and last_mx != part
            and ALL_KNOWN_COLORS.get(last_mx, "UNKNOWN") != color
            and ALL_KNOWN_COLORS.get(last_mx, "UNKNOWN") not in ("UNKNOWN", "NEEDS_PURGE")
            and color != "UNKNOWN"
        )
        new_rows.append({
            "Part": part, "Color": color, "Category": category,
            "Fixed_Machine": fixed_m or "—", "Fixed_Used": "N/A — expansion",
            "Machine": mx, "Run_Hours": round(run_x, 3),
            "Changeover_Hrs": round(cox, 3),
            "Total_Hrs_Used": round(cox + run_x, 3),
            "Rate_Per_Hour": round(r_val, 2), "Production_Qty": qty_x,
            "Indent_Daily": round(indent_daily.get(part, 0.0), 2),
            "Demand_Daily": round(demand_daily_raw.get(part, 0.0), 2),
            "Effective_Daily": round(daily, 2),
            "Demand_Driver": demand_driver(part),
            "Monthly_Indent": round(monthly, 0), "Daily_Indent": round(daily, 2),
            "Today_Target": round(today_target_qty.get(part, 0), 0),
            "Changeover": "No" if cox == 0 else "Yes",
            "Color_Purge": "Yes" if purge_x else "No",
            "Type": "Tool-Expansion",
            "Role": f"Tool-Expansion (tool {tools_used}/{tools})",
            "Tools_Available": tools, "Tools_Used": tools_used,
            "Runner_Lock": "No", "Priority_Score": score, "Phase": 2,
            "Indent_Met": (
                "YES" if indent_met_here
                else f"NO — shortfall remains (need {daily:.2f}/d)"
            ),
            "Stagger_Adjusted": "No",
        })
        print(f"      TOOL-EXP {part:26s} tool {tools_used}/{tool_hard_cap} -> {mx:15s}  {run_x:.2f}h  qty={qty_x:.0f}")

    if new_rows:
        _do_inv_build(part, new_rows[0]["Machine"], scenario_id, machine_hours,
                      current_inventory, new_rows, r_val, daily)
    for row in new_rows:
        row["Tools_Used"] = tools_used
    return new_rows

def _do_inv_build(part, machine, scenario_id, machine_hours,
                  current_inventory, rows_to_extend, r_val, daily):
    """
    Extend an existing row on `machine` to build inventory up to
    effective_opd_cap_qty.  `daily` here is effective_daily.
    """
    inv_after  = current_inventory.get(part, 0)
    cap_qty    = effective_opd_cap_qty(part, scenario_id, inv_after)
    headroom_qty = max(0.0, cap_qty - inv_after)
    if headroom_qty <= 0:
        return
    target_row = None
    if isinstance(rows_to_extend, list):
        for row in rows_to_extend:
            if isinstance(row, dict) and row.get("Part") == part and row.get("Machine") == machine:
                target_row = row
                break
    if target_row is None:
        return
    free_m = round(AVAILABLE_HOURS - machine_hours.get(machine, 0), 4)
    if free_m < 0.05:
        return
    extend_hrs = min(free_m, headroom_qty / r_val if r_val > 0 else 0)
    if extend_hrs < 0.05:
        return
    extra_qty = round(extend_hrs * r_val, 0)
    target_row["Run_Hours"] = round(float(target_row["Run_Hours"]) + extend_hrs, 3)
    target_row["Total_Hrs_Used"] = round(
        float(target_row["Changeover_Hrs"]) + float(target_row["Run_Hours"]), 3)
    target_row["Production_Qty"] = round(float(target_row["Production_Qty"]) + extra_qty, 0)
    target_row["Type"] = str(target_row["Type"]) + "+InvBuild"
    machine_hours[machine] = round(machine_hours.get(machine, 0) + extend_hrs, 4)
    current_inventory[part] = round(current_inventory.get(part, 0) + extra_qty, 0)
    print(f"      INV-BUILD {part:25s} on {machine:15s}  +{extend_hrs:.2f}h  qty+={extra_qty:.0f}  driver={demand_driver(part)}")

# =============================================================
# SECTION 17 — RUNNER PRIORITY ENFORCEMENT  (DEMAND-AWARE)
# =============================================================

def enforce_runner_priority(plan, machine_hours, machine_last_part,
                             current_inventory, already_planned,
                             priority_scores, inventory_start_of_day):
    print(f"\n{'─'*65}")
    print(f"  RUNNER PRIORITY ENFORCEMENT (threshold: < {RUNNER_PRIORITY_DAYS} days)")
    print(f"{'─'*65}")
    runner_priority_log = []

    critical_runners = []
    for part in vt_compat:
        if part in already_planned:
            continue
        if part_category.get(part, "Stranger") != "Runner":
            continue
        daily = effective_daily(part)
        r_val = rate.get(part, 1)
        if daily <= 0 or r_val <= 0:
            continue
        inv_now = current_inventory.get(part, 0)
        days_now = inv_now / daily if daily > 0 else 999
        if days_now >= RUNNER_PRIORITY_DAYS:
            continue
        skip, _ = should_skip(part)
        if skip:
            continue
        if not vt_compat.get(part):
            continue
        critical_runners.append(part)

    if not critical_runners:
        print(f"  No critical unplanned Runners found")
        return runner_priority_log

    critical_runners.sort(key=lambda p: effective_daily(p), reverse=True)
    print(f"  Critical Runners: {len(critical_runners)}")

    for runner in critical_runners:
        r_daily  = effective_daily(runner)
        r_inv    = current_inventory.get(runner, 0)
        r_inv_sod = inventory_start_of_day.get(runner, r_inv)
        r_rate   = rate.get(runner, 1)
        r_color  = part_color.get(runner, "UNKNOWN")
        r_score  = priority_scores.get(runner, 0)
        r_monthly = effective_monthly(runner)
        fixed_m  = part_fixed_machine.get(runner)
        r_days_now = r_inv / r_daily if r_daily > 0 else 0
        r_days_sod = r_inv_sod / r_daily if r_daily > 0 else 0

        shortfall_qty = max(0.0, r_daily - r_inv)
        hours_needed = max(MIN_RUN_HOURS, shortfall_qty / r_rate if r_rate > 0 else MIN_RUN_HOURS)

        machines_runner_on = {row["Machine"] for row in plan if row["Part"] == runner}
        if len(machines_runner_on) >= tools_available.get(runner, 1):
            runner_priority_log.append({
                "Runner_Part": runner, "Demand_Driver": demand_driver(runner),
                "Runner_Days_SOD": round(r_days_sod, 2),
                "Runner_Days_Now": round(r_days_now, 2), "Fixed_Machine": fixed_m or "—",
                "Fixed_Used": "FAILED — tool cap", "Runner_Daily_Indent": round(r_daily, 2),
                "Runner_Inv_Before_SOD": round(r_inv_sod, 0),
                "Runner_Shortfall": round(shortfall_qty, 0),
                "Machine_Assigned": "—", "Hours_Needed": round(hours_needed, 3),
                "Hours_Assigned": 0, "Qty_Produced": 0, "Displacement_Used": "N/A",
                "Victims": "—", "Total_Hours_Reclaimed": "—",
                "Result": f"FAILED — all tools deployed",
            })
            continue

        compatible_machines = [m for m in vt_compat.get(runner, [])]
        if fixed_m and fixed_m in compatible_machines:
            ordered_machines = [fixed_m] + [m for m in compatible_machines if m != fixed_m]
        else:
            ordered_machines = compatible_machines

        best_machine   = None
        best_co_hrs    = 0.0
        best_victims   = []
        best_disruption = float("inf")

        for m in ordered_machines:
            co_hrs   = _co_hrs_for(runner, m, machine_last_part)
            used_hrs = machine_hours.get(m, 0)
            free_hrs = round(AVAILABLE_HOURS - used_hrs, 4)
            eff_free = round(free_hrs - co_hrs, 4)

            if eff_free >= hours_needed:
                run_h = max(MIN_RUN_HOURS, min(eff_free, hours_needed))
                qty = round(run_h * r_rate, 0)
                fixed_used = (fixed_m is not None and m == fixed_m)
                machine_hours[m] = round(used_hrs + co_hrs + run_h, 4)
                current_inventory[runner] = round(current_inventory.get(runner, 0) + qty, 0)
                machine_last_part[m] = runner
                already_planned.add(runner)
                purge = co_hrs > vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS) + 0.001
                plan.append({
                    "Part": runner, "Color": r_color, "Category": "Runner",
                    "Fixed_Machine": fixed_m or "—",
                    "Fixed_Used": "YES" if fixed_used else ("FALLBACK" if fixed_m else "N/A"),
                    "Machine": m, "Run_Hours": round(run_h, 3),
                    "Changeover_Hrs": round(co_hrs, 3),
                    "Total_Hrs_Used": round(co_hrs + run_h, 3),
                    "Rate_Per_Hour": round(r_rate, 2), "Production_Qty": qty,
                    "Indent_Daily": round(indent_daily.get(runner, 0.0), 2),
                    "Demand_Daily": round(demand_daily_raw.get(runner, 0.0), 2),
                    "Effective_Daily": round(r_daily, 2),
                    "Demand_Driver": demand_driver(runner),
                    "Monthly_Indent": round(r_monthly, 0), "Daily_Indent": round(r_daily, 2),
                    "Today_Target": round(today_target_qty.get(runner, 0), 0),
                    "Changeover": "No" if co_hrs == 0 else "Yes",
                    "Color_Purge": "Yes" if purge else "No",
                    "Type": "Runner-Priority (free capacity)",
                    "Role": "Primary", "Tools_Available": tools_available.get(runner, 1),
                    "Tools_Used": 1, "Runner_Lock": "No",
                    "Priority_Score": r_score, "Phase": 1,
                    "Indent_Met": (
                        "YES" if qty >= shortfall_qty
                        else f"NO — need {r_daily:.2f}/d"
                    ),
                    "Stagger_Adjusted": "No",
                })
                runner_priority_log.append({
                    "Runner_Part": runner, "Demand_Driver": demand_driver(runner),
                    "Runner_Days_SOD": round(r_days_sod, 2),
                    "Runner_Days_Now": round(r_days_now, 2), "Fixed_Machine": fixed_m or "—",
                    "Fixed_Used": "YES" if fixed_used else "N/A",
                    "Runner_Daily_Indent": round(r_daily, 2),
                    "Runner_Inv_Before_SOD": round(r_inv_sod, 0),
                    "Runner_Shortfall": round(shortfall_qty, 0),
                    "Machine_Assigned": m, "Hours_Needed": round(hours_needed, 3),
                    "Hours_Assigned": round(run_h, 3), "Qty_Produced": qty,
                    "Displacement_Used": "No — free capacity", "Victims": "—",
                    "Total_Hours_Reclaimed": "—", "Result": "PLANNED — free capacity",
                })
                print(f"    {runner} placed on {m} (free capacity)  run={run_h:.2f}h  qty={qty:.0f}  driver={demand_driver(runner)}")
                best_machine = "DONE"
                break

            machine_rows = [r for r in plan if r["Machine"] == m]
            yieldable = [
                r for r in machine_rows
                if part_category.get(r["Part"], "Stranger") in ("Stranger", "Repeater")
                and (current_inventory.get(r["Part"], 0) / effective_daily(r["Part"])
                     if effective_daily(r["Part"]) > 0 else 999) > r_days_now
            ]
            if not yieldable:
                continue
            yieldable.sort(key=lambda r: (effective_daily(r["Part"]), -float(r.get("Production_Qty", 0))))
            reclaimable_detail = []
            for row in yieldable:
                row_run = float(row.get("Run_Hours", 0))
                if row_run <= 0:
                    continue
                if row_run > MIN_RUN_HOURS:
                    reclaimable_detail.append((row, round(row_run - MIN_RUN_HOURS, 4), "partial"))
                else:
                    reclaimable_detail.append((row, round(row_run, 4), "full_remove"))
            total_reclaimable = sum(x[1] for x in reclaimable_detail)
            runner_eff_after_reclaim = round(free_hrs + total_reclaimable - co_hrs, 4)
            if runner_eff_after_reclaim < max(MIN_RUN_HOURS, hours_needed):
                continue
            disruption = total_reclaimable
            if disruption < best_disruption:
                best_disruption = disruption
                best_machine    = m
                best_co_hrs     = co_hrs
                best_victims    = reclaimable_detail

        if best_machine is None:
            runner_priority_log.append({
                "Runner_Part": runner, "Demand_Driver": demand_driver(runner),
                "Runner_Days_SOD": round(r_days_sod, 2),
                "Runner_Days_Now": round(r_days_now, 2), "Fixed_Machine": fixed_m or "—",
                "Fixed_Used": "FAILED", "Runner_Daily_Indent": round(r_daily, 2),
                "Runner_Inv_Before_SOD": round(r_inv_sod, 0),
                "Runner_Shortfall": round(shortfall_qty, 0),
                "Machine_Assigned": "—", "Hours_Needed": round(hours_needed, 3),
                "Hours_Assigned": 0, "Qty_Produced": 0, "Displacement_Used": "N/A",
                "Victims": "—", "Total_Hours_Reclaimed": "—",
                "Result": "FAILED — no eligible machine",
            })
            continue
        if best_machine == "DONE":
            continue

        hours_to_free   = hours_needed
        victim_log_parts = []
        total_reclaimed  = 0.0
        for (victim_row, reclaimable_hrs, reclaim_type) in best_victims:
            if hours_to_free <= 0.001:
                break
            vpart    = victim_row["Part"]
            v_rate   = rate.get(vpart, 1)
            v_run_orig = float(victim_row.get("Run_Hours", 0))
            carve_hrs  = round(min(reclaimable_hrs, hours_to_free), 4)
            if carve_hrs <= 0:
                continue
            new_run_hrs = round(v_run_orig - carve_hrs, 4)
            if new_run_hrs < MIN_RUN_HOURS:
                plan.remove(victim_row)
                machine_hours[best_machine] = round(machine_hours.get(best_machine, 0) - v_run_orig, 4)
                lost_qty = round(v_run_orig * v_rate, 0)
                current_inventory[vpart] = round(current_inventory.get(vpart, 0) - lost_qty, 0)
                actually_freed = v_run_orig
                victim_log_parts.append(f"{vpart} REMOVED")
            else:
                lost_qty = round(carve_hrs * v_rate, 0)
                new_qty  = round(new_run_hrs * v_rate, 0)
                victim_row["Run_Hours"] = new_run_hrs
                victim_row["Production_Qty"] = new_qty
                victim_row["Total_Hrs_Used"] = round(float(victim_row.get("Changeover_Hrs", 0)) + new_run_hrs, 3)
                victim_row["Type"] = str(victim_row.get("Type", "")) + " [YIELDED]"
                machine_hours[best_machine] = round(machine_hours.get(best_machine, 0) - carve_hrs, 4)
                current_inventory[vpart] = round(current_inventory.get(vpart, 0) - lost_qty, 0)
                actually_freed = carve_hrs
                victim_log_parts.append(f"{vpart} -{carve_hrs:.2f}h")
            hours_to_free   = round(hours_to_free - actually_freed, 4)
            total_reclaimed = round(total_reclaimed + actually_freed, 4)

        used_now = machine_hours.get(best_machine, 0)
        free_now = round(AVAILABLE_HOURS - used_now, 4)
        eff_free = round(free_now - best_co_hrs, 4)
        if eff_free < MIN_RUN_HOURS:
            runner_priority_log.append({
                "Runner_Part": runner, "Demand_Driver": demand_driver(runner),
                "Runner_Days_SOD": round(r_days_sod, 2),
                "Runner_Days_Now": round(r_days_now, 2), "Fixed_Machine": fixed_m or "—",
                "Fixed_Used": "FAILED", "Runner_Daily_Indent": round(r_daily, 2),
                "Runner_Inv_Before_SOD": round(r_inv_sod, 0),
                "Runner_Shortfall": round(shortfall_qty, 0),
                "Machine_Assigned": best_machine, "Hours_Needed": round(hours_needed, 3),
                "Hours_Assigned": 0, "Qty_Produced": 0, "Displacement_Used": "Yes",
                "Victims": "; ".join(victim_log_parts),
                "Total_Hours_Reclaimed": round(total_reclaimed, 3),
                "Result": "FAILED — safety check after carve",
            })
            continue

        run_hrs  = max(MIN_RUN_HOURS, min(eff_free, hours_needed))
        qty      = round(run_hrs * r_rate, 0)
        fixed_used = (fixed_m is not None and best_machine == fixed_m)
        purge = best_co_hrs > vt_changeover.get(best_machine, DEFAULT_CHANGEOVER_HRS) + 0.001
        machine_hours[best_machine] = round(used_now + best_co_hrs + run_hrs, 4)
        current_inventory[runner]   = round(current_inventory.get(runner, 0) + qty, 0)
        machine_last_part[best_machine] = runner
        already_planned.add(runner)

        plan.append({
            "Part": runner, "Color": r_color, "Category": "Runner",
            "Fixed_Machine": fixed_m or "—",
            "Fixed_Used": "YES" if fixed_used else ("FALLBACK" if fixed_m else "N/A"),
            "Machine": best_machine, "Run_Hours": round(run_hrs, 3),
            "Changeover_Hrs": round(best_co_hrs, 3),
            "Total_Hrs_Used": round(best_co_hrs + run_hrs, 3),
            "Rate_Per_Hour": round(r_rate, 2), "Production_Qty": qty,
            "Indent_Daily": round(indent_daily.get(runner, 0.0), 2),
            "Demand_Daily": round(demand_daily_raw.get(runner, 0.0), 2),
            "Effective_Daily": round(r_daily, 2),
            "Demand_Driver": demand_driver(runner),
            "Monthly_Indent": round(r_monthly, 0), "Daily_Indent": round(r_daily, 2),
            "Today_Target": round(today_target_qty.get(runner, 0), 0),
            "Changeover": "No" if best_co_hrs == 0 else "Yes",
            "Color_Purge": "Yes" if purge else "No",
            "Type": "Runner-Priority [DISPLACED]",
            "Role": "Primary", "Tools_Available": tools_available.get(runner, 1),
            "Tools_Used": 1, "Runner_Lock": "No",
            "Priority_Score": r_score, "Phase": 1,
            "Indent_Met": (
                "YES" if qty >= shortfall_qty
                else f"NO — need {r_daily:.2f}/d"
            ),
            "Stagger_Adjusted": "No",
        })
        runner_priority_log.append({
            "Runner_Part": runner, "Demand_Driver": demand_driver(runner),
            "Runner_Days_SOD": round(r_days_sod, 2),
            "Runner_Days_Now": round(r_days_now, 2), "Fixed_Machine": fixed_m or "—",
            "Fixed_Used": "YES" if fixed_used else "N/A",
            "Runner_Daily_Indent": round(r_daily, 2),
            "Runner_Inv_Before_SOD": round(r_inv_sod, 0),
            "Runner_Shortfall": round(shortfall_qty, 0),
            "Machine_Assigned": best_machine, "Hours_Needed": round(hours_needed, 3),
            "Hours_Assigned": round(run_hrs, 3), "Qty_Produced": qty,
            "Displacement_Used": "Yes", "Victims": "; ".join(victim_log_parts),
            "Total_Hours_Reclaimed": round(total_reclaimed, 3),
            "Result": "PLANNED — displacement successful" if qty >= shortfall_qty - 0.5 else "PLANNED — partial",
        })
        print(f"    {runner:30s} -> {best_machine}  run={run_hrs:.2f}h  qty={qty:.0f}  driver={demand_driver(runner)}")

    return runner_priority_log

# =============================================================
# SECTION 18 — DISPLACEMENT PRE-PASS  (DEMAND-AWARE)
# =============================================================

def displace_for_zero_inv(part, machine_hours, machine_last_part,
                           current_inventory, plan, already_planned, priority_scores):
    daily    = effective_daily(part)
    r_val    = rate.get(part, 1)
    category = part_category.get(part, "Stranger")
    score    = priority_scores.get(part, 0)
    compatible = vt_compat.get(part, [])
    fixed_m  = part_fixed_machine.get(part)
    if not compatible:
        return False
    candidate_machines = [
        m for m in compatible
        if m not in _phase_a_machines
        and round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4) < MIN_RUN_HOURS
    ]
    if not candidate_machines:
        return False

    best_machine    = None
    best_victim_row = None
    best_victim_days = -1
    part_days_cov   = current_inventory.get(part, 0) / daily if daily > 0 else 0

    for m in candidate_machines:
        for row in [r for r in plan if r["Machine"] == m]:
            vpart  = row["Part"]
            vdaily = effective_daily(vpart)
            vinv   = current_inventory.get(vpart, 0)
            vdays  = vinv / vdaily if vdaily > 0 else 999
            if vdays <= part_days_cov:
                continue
            if vdays < SAFETY_DAYS or vinv <= 0:
                continue
            vrun = float(row.get("Run_Hours", 0))
            if vrun - MIN_RUN_HOURS < MIN_RUN_HOURS:
                continue
            if vdays > best_victim_days:
                best_victim_days = vdays
                best_victim_row  = row
                best_machine     = m

    if best_machine is None or best_victim_row is None:
        return False

    vpart    = best_victim_row["Part"]
    vr_val   = rate.get(vpart, 1)
    reduce_h = MIN_RUN_HOURS
    lost_qty = round(reduce_h * vr_val, 0)
    best_victim_row["Run_Hours"]      = round(float(best_victim_row["Run_Hours"]) - reduce_h, 3)
    best_victim_row["Production_Qty"] = round(float(best_victim_row["Production_Qty"]) - lost_qty, 0)
    best_victim_row["Total_Hrs_Used"] = round(
        float(best_victim_row.get("Changeover_Hrs", 0)) + float(best_victim_row["Run_Hours"]), 3)
    best_victim_row["Type"] = str(best_victim_row.get("Type", "")) + " [DISPLACED]"
    current_inventory[vpart] = round(current_inventory.get(vpart, 0) - lost_qty, 0)
    machine_hours[best_machine] = round(machine_hours.get(best_machine, 0) - reduce_h, 4)

    p_col  = part_color.get(part, "UNKNOWN")
    last   = machine_last_part.get(best_machine)
    l_col  = ALL_KNOWN_COLORS.get(last, "UNKNOWN") if last else "UNKNOWN"
    if last is None or last == part:
        co_hrs = 0.0
    else:
        base_co = vt_changeover.get(best_machine, DEFAULT_CHANGEOVER_HRS)
        purge   = COLOR_PURGE_HRS if p_col != l_col and p_col != "UNKNOWN" and l_col not in ("UNKNOWN", "NEEDS_PURGE") else 0.0
        co_hrs  = base_co + purge

    eff_free    = round(AVAILABLE_HOURS - machine_hours.get(best_machine, 0) - co_hrs, 4)
    hrs_for_indent = max(MIN_RUN_HOURS, daily / r_val if r_val > 0 else MIN_RUN_HOURS)
    run_hrs     = max(MIN_RUN_HOURS, min(eff_free, hrs_for_indent))
    qty         = round(run_hrs * r_val, 0)

    machine_hours[best_machine]     = round(machine_hours.get(best_machine, 0) + co_hrs + run_hrs, 4)
    current_inventory[part]         = round(current_inventory.get(part, 0) + qty, 0)
    machine_last_part[best_machine] = part
    already_planned.add(part)

    has_purge  = co_hrs > vt_changeover.get(best_machine, DEFAULT_CHANGEOVER_HRS) + 0.001
    fixed_used = (fixed_m is not None and best_machine == fixed_m)

    plan.append({
        "Part": part, "Color": p_col, "Category": category,
        "Fixed_Machine": fixed_m or "—",
        "Fixed_Used": "YES" if fixed_used else ("FALLBACK" if fixed_m else "N/A"),
        "Machine": best_machine, "Run_Hours": round(run_hrs, 3),
        "Changeover_Hrs": round(co_hrs, 3),
        "Total_Hrs_Used": round(co_hrs + run_hrs, 3),
        "Rate_Per_Hour": round(r_val, 2), "Production_Qty": qty,
        "Indent_Daily": round(indent_daily.get(part, 0.0), 2),
        "Demand_Daily": round(demand_daily_raw.get(part, 0.0), 2),
        "Effective_Daily": round(daily, 2),
        "Demand_Driver": demand_driver(part),
        "Monthly_Indent": round(effective_monthly(part), 0),
        "Daily_Indent": round(daily, 2),
        "Today_Target": round(today_target_qty.get(part, 0), 0),
        "Changeover": "No" if co_hrs == 0 else "Yes",
        "Color_Purge": "Yes" if has_purge else "No",
        "Type": "Displacement [ZERO-INV]",
        "Role": "Primary", "Tools_Available": tools_available.get(part, 1),
        "Tools_Used": 1, "Runner_Lock": "No", "Priority_Score": score,
        "Phase": 1,
        "Indent_Met": (
            "YES" if qty >= daily
            else f"NO — need {daily:.2f}/d"
        ),
        "Stagger_Adjusted": "No",
    })
    print(f"      DISPLACEMENT  {part:26s} -> {best_machine:15s}  run={run_hrs:.2f}h  qty={qty:.0f}  driver={demand_driver(part)}")
    return True

# =============================================================
# SECTION 19 — TOOL-CHANGER HELPERS
# =============================================================

def _collect_co_events(plan, machines):
    events = []
    for m in machines:
        m_rows = [r for r in plan if r["Machine"] == m]
        if len(m_rows) < 2:
            continue
        cursor = 0.0
        for i, row in enumerate(m_rows):
            co_h  = float(row.get("Changeover_Hrs") or 0.0)
            run_h = float(row.get("Run_Hours") or 0.0)
            if co_h > 0 and i > 0:
                events.append({
                    "machine": m, "part_before": m_rows[i-1]["Part"],
                    "part_after": row["Part"], "co_duration": co_h,
                    "natural_start": cursor, "row_before": m_rows[i-1],
                    "row_after": row, "actual_start": None, "wait_hrs": 0.0,
                })
            cursor += co_h + run_h
    return events

def _recompute_natural_start(ev, plan):
    m, target = ev["machine"], ev["row_after"]
    cursor = 0.0
    for r in [r for r in plan if r["Machine"] == m]:
        if r is target:
            break
        cursor += float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0)
    return cursor

def _machine_spare(m, plan):
    used = sum(float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0) for r in plan if r["Machine"] == m)
    return max(0.0, AVAILABLE_HOURS - used)

def _extend_row_before(ev, wait_hrs, plan, machine_hours):
    m      = ev["machine"]
    spare  = _machine_spare(m, plan)
    extend_by = min(wait_hrs, spare)
    if extend_by <= 0:
        return 0.0, 0
    rb    = ev["row_before"]
    r_val = rate.get(rb["Part"], 1.0)
    extra = round(extend_by * r_val, 0)
    rb["Run_Hours"]      = round(float(rb.get("Run_Hours") or 0) + extend_by, 3)
    rb["Production_Qty"] = round(float(rb.get("Production_Qty") or 0) + extra, 0)
    rb["Total_Hrs_Used"] = round(float(rb.get("Changeover_Hrs") or 0) + float(rb["Run_Hours"]), 3)
    rb["Stagger_Adjusted"] = f"CO: extended +{round(extend_by * 60, 1)}min"
    machine_hours[m] = round(machine_hours.get(m, 0) + extend_by, 4)
    return extend_by, extra

# =============================================================
# SECTION 20 — CO STAGGER  (DEMAND-AWARE)
# =============================================================

MIN_CO_GAP_HRS = 20 / 60.0

def _finish_time_of_co(ev, plan):
    m, target = ev["machine"], ev["row_after"]
    total = 0.0
    for row in plan:
        if row["Machine"] != m:
            continue
        if row is target:
            break
        total += float(row.get("Run_Hours") or 0) + float(row.get("Changeover_Hrs") or 0)
    return round(total, 4)

def stagger_co_by_quantity(plan, machines, scenario_id, current_inventory):
    events = _collect_co_events(plan, machines)
    if len(events) < 2:
        return 0
    adjustments = 0
    max_passes = len(events) * 2
    for _ in range(max_passes):
        for ev in events:
            ev["_ft"] = _finish_time_of_co(ev, plan)
        events.sort(key=lambda e: e["_ft"])
        conflict = None
        for i in range(len(events) - 1):
            required = events[i]["co_duration"] + MIN_CO_GAP_HRS
            gap      = events[i+1]["_ft"] - events[i]["_ft"]
            if gap < required - 0.001:
                conflict = (events[i], events[i+1], gap, required)
                break
        if conflict is None:
            break
        ev_early, ev_late, gap, required_gap = conflict
        shortfall_hrs = required_gap - gap
        row_late  = ev_late["row_before"]
        p_late    = row_late["Part"]
        r_late    = rate.get(p_late, 1)
        m_late    = ev_late["machine"]
        daily_l   = effective_daily(p_late)
        inv_l     = current_inventory.get(p_late, 0)
        cap_qty   = effective_opd_cap_qty(p_late, scenario_id, inv_l)
        headroom  = max(0.0, cap_qty - inv_l)
        push_qty  = round(shortfall_hrs * r_late, 0)
        used_m    = sum(float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0) for r in plan if r["Machine"] == m_late)
        free_m    = max(0.0, AVAILABLE_HOURS - used_m)
        can_push  = push_qty <= headroom and shortfall_hrs <= free_m + 0.001 and r_late > 0 and m_late not in _phase_a_machines

        if can_push:
            row_late["Run_Hours"]      = round(float(row_late.get("Run_Hours") or 0) + shortfall_hrs, 3)
            row_late["Production_Qty"] = round(float(row_late.get("Production_Qty") or 0) + push_qty, 0)
            row_late["Total_Hrs_Used"] = round(float(row_late.get("Changeover_Hrs") or 0) + float(row_late["Run_Hours"]), 3)
            current_inventory[p_late]  = round(current_inventory.get(p_late, 0) + push_qty, 0)
            adjustments += 1
            continue

        row_early = ev_early["row_before"]
        p_early   = row_early["Part"]
        r_early   = rate.get(p_early, 1)
        daily_e   = effective_daily(p_early)
        inv_e     = current_inventory.get(p_early, 0)
        produced_e = float(row_early.get("Production_Qty") or 0)
        min_qty_e  = max(0.0, daily_e - inventory.get(p_early, 0))
        max_pull   = max(0.0, produced_e - min_qty_e)
        pull_qty   = round(shortfall_hrs * r_early, 0)
        can_pull   = (pull_qty <= max_pull and r_early > 0
                      and produced_e - pull_qty >= MIN_RUN_HOURS * r_early
                      and ev_early["machine"] not in _phase_a_machines)

        if can_pull:
            row_early["Run_Hours"]      = round(float(row_early.get("Run_Hours") or 0) - shortfall_hrs, 3)
            row_early["Production_Qty"] = round(produced_e - pull_qty, 0)
            row_early["Total_Hrs_Used"] = round(float(row_early.get("Changeover_Hrs") or 0) + float(row_early["Run_Hours"]), 3)
            current_inventory[p_early]  = round(current_inventory.get(p_early, 0) - pull_qty, 0)
            adjustments += 1
            continue
        break
    return adjustments

def stagger_changeovers(plan, machines, machine_hours):
    print(f"\n  Tool-Changer Serial Queue Scheduler")
    events = _collect_co_events(plan, machines)
    if not events:
        print(f"  No changeovers — tool changer idle")
        return
    events.sort(key=lambda e: e["natural_start"])
    if len(events) > MAX_DAILY_CO:
        events = events[:MAX_DAILY_CO]
    tool_changer_free_at = 0.0
    total_extra_pcs = 0
    for idx, ev in enumerate(events, 1):
        natural_start = _recompute_natural_start(ev, plan)
        co_h          = ev["co_duration"]
        actual_start  = max(natural_start, tool_changer_free_at)
        wait_hrs      = round(actual_start - natural_start, 4)
        tool_changer_free_at = actual_start + co_h
        if wait_hrs > 0.001:
            _, extra_pcs = _extend_row_before(ev, wait_hrs, plan, machine_hours)
            total_extra_pcs += extra_pcs
        ev["actual_start"] = actual_start
        ev["wait_hrs"]     = wait_hrs
    print(f"  {len(events)} CO events processed  |  Extra pcs from wait fill: {total_extra_pcs}")

# =============================================================
# SECTION 21 — 22H UTILISATION ENFORCER  (DEMAND-AWARE)
# =============================================================

def _add_part_to_machine(p, m, run_hrs, co_hrs, machine_hours,
                          machine_last_part, current_inventory,
                          already_planned, plan, priority_scores,
                          type_label, role_label):
    r_val   = rate.get(p, 1)
    qty     = round(run_hrs * r_val, 0)
    last    = machine_last_part.get(m)
    p_color = part_color.get(p, "UNKNOWN")
    l_color = ALL_KNOWN_COLORS.get(last, "UNKNOWN") if last else "UNKNOWN"
    has_purge = (last is not None and last != p and p_color != l_color
                 and p_color != "UNKNOWN" and l_color not in ("UNKNOWN", "NEEDS_PURGE"))
    machine_hours[m]      = round(machine_hours.get(m, 0) + co_hrs + run_hrs, 4)
    current_inventory[p]  = round(current_inventory.get(p, 0) + qty, 0)
    machine_last_part[m]  = p
    already_planned.add(p)
    plan.append({
        "Part": p, "Color": p_color, "Category": part_category.get(p, "Stranger"),
        "Fixed_Machine": part_fixed_machine.get(p, "—"),
        "Fixed_Used": "N/A — enforcer", "Machine": m,
        "Run_Hours": round(run_hrs, 3), "Changeover_Hrs": round(co_hrs, 3),
        "Total_Hrs_Used": round(co_hrs + run_hrs, 3),
        "Rate_Per_Hour": round(r_val, 2), "Production_Qty": qty,
        "Indent_Daily": round(indent_daily.get(p, 0.0), 2),
        "Demand_Daily": round(demand_daily_raw.get(p, 0.0), 2),
        "Effective_Daily": round(effective_daily(p), 2),
        "Demand_Driver": demand_driver(p),
        "Monthly_Indent": round(effective_monthly(p), 0),
        "Daily_Indent": round(effective_daily(p), 2),
        "Today_Target": round(today_target_qty.get(p, 0), 0),
        "Changeover": "No" if co_hrs == 0 else "Yes",
        "Color_Purge": "Yes" if has_purge else "No",
        "Type": type_label, "Role": role_label,
        "Tools_Available": tools_available.get(p, 1),
        "Tools_Used": 1, "Runner_Lock": "No",
        "Priority_Score": round(priority_scores.get(p, 0), 2),
        "Phase": 1, "Stagger_Adjusted": "No",
    })
    return qty

def _extend_existing_on_machine(m, plan, machine_hours, current_inventory,
                                 priority_scores, ceiling_days, remaining):
    consumed = 0.0
    parts_on_m = sorted(
        [row for row in plan if row["Machine"] == m],
        key=lambda r: priority_scores.get(r["Part"], 0) if isinstance(priority_scores.get(r["Part"], 0), (int, float)) else 0,
        reverse=True,
    )
    for row in parts_on_m:
        if remaining < 0.001:
            break
        p_ext   = row["Part"]
        if is_hard_skip(p_ext):
            continue
        r_ext   = rate.get(p_ext, 1)
        daily_p = effective_daily(p_ext)
        inv_now = current_inventory.get(p_ext, 0)
        headroom = max(0.0, ceiling_days * daily_p - inv_now) if daily_p > 0 else 0
        ext_hrs  = min(remaining, headroom / r_ext if r_ext > 0 else 0)
        if ext_hrs < 0.001:
            continue
        extra_qty = round(ext_hrs * r_ext, 0)
        row["Run_Hours"]      = round(float(row.get("Run_Hours", 0)) + ext_hrs, 3)
        row["Production_Qty"] = round(float(row.get("Production_Qty", 0)) + extra_qty, 0)
        row["Total_Hrs_Used"] = round(float(row.get("Changeover_Hrs", 0)) + float(row["Run_Hours"]), 3)
        row["Type"] = str(row.get("Type", "")) + f"+Ext{ceiling_days}d"
        machine_hours[m]      = round(machine_hours.get(m, 0) + ext_hrs, 4)
        current_inventory[p_ext] = round(current_inventory.get(p_ext, 0) + extra_qty, 0)
        remaining = round(remaining - ext_hrs, 4)
        consumed += ext_hrs
    return consumed, remaining

def utilization_enforcer(plan, machine_hours, machine_last_part,
                          all_parts, already_planned,
                          current_inventory, scenario_id, priority_scores):
    print(f"\n  22H UTILIZATION ENFORCER (DEMAND-AWARE)")
    micro_idle_log = []
    floor_hrs = AVAILABLE_HOURS * (UTIL_TARGET_PCT / 100.0)

    machines_by_util = sorted(vt_machines, key=lambda m: machine_hours.get(m, 0))

    for m in machines_by_util:
        if m in _phase_a_machines:
            continue

        remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
        if remaining < 0.05:
            continue
        last_on_m  = machine_last_part.get(m)
        last_color = part_color.get(last_on_m, "UNKNOWN") if last_on_m else "UNKNOWN"

        # S0 — Extend to 90% floor
        current_util_hrs = machine_hours.get(m, 0)
        if current_util_hrs < floor_hrs:
            needed = round(floor_hrs - current_util_hrs, 4)
            _, remaining = _extend_existing_on_machine(
                m, plan, machine_hours, current_inventory,
                priority_scores, opd_cap(scenario_id), min(needed, remaining))
            remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)

        if remaining < 0.05:
            continue

        # S2 — Add new parts (demand-aware OPD cap)
        if remaining >= MIN_RUN_HOURS:
            machine_compat = machine_compatible_parts.get(m, [])
            unplanned = []
            for p in machine_compat:
                if indent_monthly.get(p, 0) <= 0 and demand_daily_raw.get(p, 0) <= 0:
                    continue
                if rate.get(p, 0) <= 0:
                    continue
                if terminal_blocked(p)[0]:
                    continue
                pfm = part_fixed_machine.get(p)
                if pfm is not None and pfm != m:
                    continue
                daily_p = effective_daily(p)
                inv_now = current_inventory.get(p, 0)
                cap_q   = effective_opd_cap_qty(p, scenario_id, inv_now)
                if daily_p > 0 and inv_now >= cap_q:
                    continue
                cat = part_category.get(p, "Stranger")
                if cat == "Runner" and p in already_planned:
                    if len({row["Machine"] for row in plan if row["Part"] == p}) >= tools_available.get(p, 1):
                        continue
                unplanned.append(p)

            def _sort_key(p):
                total_qty_today = _get_part_total_qty(p, plan)
                is_unplanned    = 0 if total_qty_today == 0 else 1
                needs_co        = 0 if (last_on_m is None or last_on_m == p) else 1
                p_col           = part_color.get(p, "UNKNOWN")
                same_col        = 0 if (needs_co == 1 and p_col == last_color and p_col != "UNKNOWN") else 1
                cat_pri         = {"Runner": 0, "Repeater": 1, "Stranger": 2}.get(part_category.get(p, "Stranger"), 2)
                inv_now_p       = current_inventory.get(p, 0)
                daily_p         = effective_daily(p)
                days_cov        = inv_now_p / daily_p if daily_p > 0 else 999
                sc              = priority_scores.get(p, 0)
                return (is_unplanned, needs_co, same_col, cat_pri, days_cov, -sc)

            unplanned.sort(key=_sort_key)

            for p in unplanned:
                if remaining < MIN_RUN_HOURS:
                    break
                co_hrs  = _co_hrs_for(p, m, machine_last_part)
                eff_free = round(remaining - co_hrs, 4)
                if eff_free < MIN_RUN_HOURS:
                    continue
                if co_hrs > 0 and _current_co_count(plan) >= MAX_DAILY_CO:
                    continue
                daily_p   = effective_daily(p)
                r_val     = rate.get(p, 1)
                inv_now_p = current_inventory.get(p, 0)
                cap_qty   = effective_opd_cap_qty(p, scenario_id, inv_now_p)
                headroom  = max(0.0, cap_qty - inv_now_p)
                if headroom <= 0:
                    continue
                shortfall = max(0.0, daily_p - inv_now_p)
                min_run   = max(MIN_RUN_HOURS, shortfall / r_val if r_val > 0 else MIN_RUN_HOURS)
                run_hrs   = max(min_run, min(eff_free, headroom / r_val if r_val > 0 else eff_free))
                was_unplanned = _get_part_total_qty(p, plan) == 0
                qty = _add_part_to_machine(
                    p, m, run_hrs, co_hrs, machine_hours, machine_last_part,
                    current_inventory, already_planned, plan, priority_scores,
                    "Filler-Unplanned" if was_unplanned else "Filler-Expand", "Primary",
                )
                remaining = round(remaining - co_hrs - run_hrs, 4)
                last_on_m  = machine_last_part.get(m)
                last_color = part_color.get(last_on_m, "UNKNOWN") if last_on_m else "UNKNOWN"
                print(f"    [S2] {p:28s} -> {m:15s}  {run_hrs:.2f}h  qty={qty:.0f}  driver={demand_driver(p)}")

        if remaining < 0.05:
            continue

        # S1 — Extend through progressive ceilings
        for ceiling in [opd_cap(scenario_id), STRATEGIC_BUFFER_DAYS, ABSOLUTE_MAX_DAYS]:
            if remaining < 0.001:
                break
            _, remaining = _extend_existing_on_machine(
                m, plan, machine_hours, current_inventory,
                priority_scores, ceiling, remaining)
            remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)

        if remaining < MIN_RUN_HOURS:
            if 0.001 < remaining:
                _, remaining = _extend_existing_on_machine(
                    m, plan, machine_hours, current_inventory,
                    priority_scores, ABSOLUTE_MAX_DAYS, remaining)
            continue

        # S3 — Runner spare tool
        indent_met_parts = _parts_with_indent_met(plan)
        runner_spare = [
            p for p in already_planned
            if m in vt_compat.get(p, [])
            and part_category.get(p, "Stranger") == "Runner"
            and p not in indent_met_parts
            and m not in [row["Machine"] for row in plan if row["Part"] == p]
            and len({row["Machine"] for row in plan if row["Part"] == p}) < tools_available.get(p, 1)
            and (part_fixed_machine.get(p) is None or part_fixed_machine.get(p) == m)
        ]
        runner_spare.sort(key=lambda p: (
            0 if (last_on_m is None or last_on_m == p) else 1,
            0 if (part_color.get(p, "UNKNOWN") == last_color and last_color != "UNKNOWN") else 1,
            -priority_scores.get(p, 0),
        ))
        for p in runner_spare:
            if remaining < MIN_RUN_HOURS:
                break
            co_hrs  = _co_hrs_for(p, m, machine_last_part)
            eff_free = round(remaining - co_hrs, 4)
            if eff_free < MIN_RUN_HOURS:
                continue
            if co_hrs > 0 and _current_co_count(plan) >= MAX_DAILY_CO:
                continue
            daily_p   = effective_daily(p)
            r_val     = rate.get(p, 1)
            inv_now_p = current_inventory.get(p, 0)
            cap_qty   = effective_opd_cap_qty(p, scenario_id, inv_now_p)
            headroom  = max(0.0, cap_qty - inv_now_p)
            if headroom <= 0:
                continue
            run_hrs = max(MIN_RUN_HOURS, min(eff_free, headroom / r_val if r_val > 0 else eff_free))
            qty     = round(run_hrs * r_val, 0)
            machine_hours[m]     = round(machine_hours.get(m, 0) + co_hrs + run_hrs, 4)
            current_inventory[p] = round(current_inventory.get(p, 0) + qty, 0)
            machine_last_part[m] = p
            remaining = round(remaining - co_hrs - run_hrs, 4)
            tools_used_now = len({row["Machine"] for row in plan if row["Part"] == p}) + 1
            plan.append({
                "Part": p, "Color": part_color.get(p, "UNKNOWN"), "Category": "Runner",
                "Fixed_Machine": part_fixed_machine.get(p, "—"),
                "Fixed_Used": "N/A — enforcer", "Machine": m,
                "Run_Hours": round(run_hrs, 3), "Changeover_Hrs": round(co_hrs, 3),
                "Total_Hrs_Used": round(co_hrs + run_hrs, 3),
                "Rate_Per_Hour": round(r_val, 2), "Production_Qty": qty,
                "Indent_Daily": round(indent_daily.get(p, 0.0), 2),
                "Demand_Daily": round(demand_daily_raw.get(p, 0.0), 2),
                "Effective_Daily": round(daily_p, 2),
                "Demand_Driver": demand_driver(p),
                "Monthly_Indent": round(effective_monthly(p), 0),
                "Daily_Indent": round(daily_p, 2),
                "Today_Target": round(today_target_qty.get(p, 0), 0),
                "Changeover": "No" if co_hrs == 0 else "Yes",
                "Color_Purge": "No",
                "Type": "Runner-Rerun",
                "Role": f"Tool-Expansion (tool {tools_used_now})",
                "Tools_Available": tools_available.get(p, 1),
                "Tools_Used": tools_used_now, "Runner_Lock": "No",
                "Priority_Score": round(priority_scores.get(p, 0), 2),
                "Phase": 3, "Stagger_Adjusted": "No",
            })
            last_on_m  = p
            last_color = part_color.get(p, "UNKNOWN")
            print(f"    [S3] {p:28s} -> {m:15s}  {run_hrs:.2f}h  tool {tools_used_now}")
            break

        if remaining < MIN_RUN_HOURS:
            remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
            if 0.001 < remaining:
                _, remaining = _extend_existing_on_machine(
                    m, plan, machine_hours, current_inventory,
                    priority_scores, ABSOLUTE_MAX_DAYS, remaining)
            continue

        # S4 — Any compatible part (demand-aware headroom)
        s4_cands = []
        machine_compat = machine_compatible_parts.get(m, [])
        for p in machine_compat:
            if terminal_blocked(p)[0]:
                continue
            if rate.get(p, 0) <= 0:
                continue
            if indent_monthly.get(p, 0) <= 0 and demand_daily_raw.get(p, 0) <= 0:
                continue
            pfm = part_fixed_machine.get(p)
            if pfm is not None and pfm != m:
                continue
            s4_cands.append(p)

        s4_cands.sort(key=lambda p: (
            0 if (last_on_m is None or last_on_m == p) else 1,
            {"Runner": 0, "Repeater": 1, "Stranger": 2}.get(part_category.get(p, "Stranger"), 2),
            -effective_daily(p),
        ))
        for p in s4_cands:
            if remaining < MIN_RUN_HOURS:
                break
            co_hrs  = _co_hrs_for(p, m, machine_last_part)
            eff_free = round(remaining - co_hrs, 4)
            if eff_free < MIN_RUN_HOURS:
                continue
            if co_hrs > 0 and _current_co_count(plan) >= MAX_DAILY_CO:
                continue
            r_val     = rate.get(p, 1)
            daily_p   = effective_daily(p)
            inv_now_p = current_inventory.get(p, 0)
            headroom  = max(0.0, ABSOLUTE_MAX_DAYS * daily_p - inv_now_p) if daily_p > 0 else (eff_free * r_val)
            if headroom <= 0:
                continue
            run_hrs = max(MIN_RUN_HOURS, min(eff_free, headroom / r_val if r_val > 0 else eff_free))
            qty = _add_part_to_machine(
                p, m, run_hrs, co_hrs, machine_hours, machine_last_part,
                current_inventory, already_planned, plan, priority_scores,
                "Filler-AnyCompatible", "Primary",
            )
            remaining = round(remaining - co_hrs - run_hrs, 4)
            last_on_m  = machine_last_part.get(m)
            last_color = part_color.get(last_on_m, "UNKNOWN") if last_on_m else "UNKNOWN"
            print(f"    [S4] {p:28s} -> {m:15s}  {run_hrs:.2f}h  qty={qty:.0f}  driver={demand_driver(p)}")

        # S5 — Final tail fill
        remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
        if 0.001 < remaining:
            _, remaining = _extend_existing_on_machine(
                m, plan, machine_hours, current_inventory,
                priority_scores, ABSOLUTE_MAX_DAYS, remaining)

        final_remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
        if final_remaining >= 0.25:
            util_final = round((1 - final_remaining / AVAILABLE_HOURS) * 100, 1)
            micro_idle_log.append({
                "Machine": m, "Idle_Hrs": round(final_remaining, 3),
                "Utilization_Pct": util_final,
                "Note": "Exhausted all compatible parts",
            })

    return micro_idle_log

# =============================================================
# SECTION 21B — VALIDATION
# =============================================================

def validate_plan_rows(plan, current_inventory):
    violations = []
    for row in plan:
        p     = row["Part"]
        run_h = float(row.get("Run_Hours", 0) or 0)
        qty   = float(row.get("Production_Qty", 0) or 0)
        daily = effective_daily(p)
        v = []
        if run_h < MIN_RUN_HOURS - 0.001:
            v.append(f"Run_Hours={run_h:.3f} < MIN={MIN_RUN_HOURS}")
        if v:
            row["Indent_Met"] = "VIOLATION: " + " | ".join(v)
            violations.append({
                "Part": p, "Machine": row.get("Machine", "—"),
                "Run_Hours": run_h, "Qty": qty,
                "Effective_Daily": daily,
                "Demand_Driver": demand_driver(p),
                "Violations": " | ".join(v),
            })
    if violations:
        print(f"\n  VALIDATION: {len(violations)} violation(s)")
    else:
        print(f"\n  Validation: all rows OK")
    return violations

# =============================================================
# SECTION 21C — STRATEGIC BUFFER FILLER  (DEMAND-AWARE)
# =============================================================

def strategic_buffer_score(part, current_inventory):
    daily = effective_daily(part)
    inv   = current_inventory.get(part, 0.0)
    r_val = rate.get(part, 1.0)
    cat   = part_category.get(part, "Stranger")
    if daily <= 0 or r_val <= 0:
        return 0.0
    velocity = daily / max(float(inv), 1.0)
    cat_w = {"Runner": 1.0, "Repeater": 0.7, "Stranger": 0.4}.get(cat, 0.4)
    return round(velocity * cat_w * STRATEGIC_PRIORITY_DISCOUNT * 100, 2)

def strategic_buffer_filler(plan, machine_hours, machine_last_part,
                              all_parts, already_planned,
                              current_inventory, scenario_id):
    print(f"\n{'─'*65}")
    print(f"  STRATEGIC BUFFER FILLER (DEMAND-AWARE)")
    print(f"{'─'*65}")
    filled_count = 0
    strat_scores = {p: strategic_buffer_score(p, current_inventory) for p in all_parts}

    machines_by_util = sorted(vt_machines, key=lambda m: machine_hours.get(m, 0))

    for m in machines_by_util:
        if m in _phase_a_machines:
            continue
        remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
        if remaining < 0.001:
            continue
        last_on_m  = machine_last_part.get(m)
        last_color = part_color.get(last_on_m, "UNKNOWN") if last_on_m else "UNKNOWN"
        co_cap_hit = _current_co_count(plan) >= MAX_DAILY_CO

        # Step A: Extend existing
        for ceiling_days in [STRATEGIC_BUFFER_DAYS, ABSOLUTE_MAX_DAYS]:
            if remaining < 0.001:
                break
            consumed, remaining = _extend_existing_on_machine(
                m, plan, machine_hours, current_inventory,
                strat_scores, ceiling_days, remaining)
            if consumed > 0:
                filled_count += 1
            remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)

        if remaining < MIN_RUN_HOURS:
            if remaining > 0.001:
                _, remaining = _extend_existing_on_machine(
                    m, plan, machine_hours, current_inventory,
                    strat_scores, ABSOLUTE_MAX_DAYS, remaining)
            continue

        # Step B: Add NEW parts
        if not co_cap_hit:
            machine_compat = machine_compatible_parts.get(m, [])
            candidates = []
            for p in machine_compat:
                if terminal_blocked(p)[0]:
                    continue
                if rate.get(p, 0) <= 0:
                    continue
                if indent_monthly.get(p, 0) <= 0 and demand_daily_raw.get(p, 0) <= 0:
                    continue
                pfm = part_fixed_machine.get(p)
                if pfm is not None and pfm != m:
                    continue
                cat = part_category.get(p, "Stranger")
                if cat == "Runner" and p in already_planned:
                    if len({row["Machine"] for row in plan if row["Part"] == p}) >= tools_available.get(p, 1):
                        continue
                inv_p   = current_inventory.get(p, 0)
                daily_p = effective_daily(p)
                if daily_p > 0 and inv_p >= ABSOLUTE_MAX_DAYS * daily_p:
                    continue
                candidates.append(p)

            def _strat_sort(p):
                total_qty_today = _get_part_total_qty(p, plan)
                is_unplanned    = 0 if total_qty_today == 0 else 1
                needs_co        = 0 if (last_on_m is None or last_on_m == p) else 1
                p_col           = part_color.get(p, "UNKNOWN")
                same_col        = 0 if (needs_co == 1 and p_col == last_color and p_col != "UNKNOWN") else 1
                sc              = strat_scores.get(p, 0)
                return (is_unplanned, needs_co, same_col, -sc)

            candidates.sort(key=_strat_sort)

            for p in candidates:
                if remaining < MIN_RUN_HOURS:
                    break
                co_hrs  = _co_hrs_for(p, m, machine_last_part)
                eff_free = round(remaining - co_hrs, 4)
                if eff_free < MIN_RUN_HOURS:
                    continue
                if co_hrs > 0 and _current_co_count(plan) >= MAX_DAILY_CO:
                    continue
                r_val     = rate.get(p, 1)
                daily_p   = effective_daily(p)
                inv_now   = current_inventory.get(p, 0)
                headroom  = max(0.0, STRATEGIC_BUFFER_DAYS * daily_p - inv_now)
                if headroom <= 0:
                    headroom = max(0.0, ABSOLUTE_MAX_DAYS * daily_p - inv_now)
                if headroom <= 0:
                    continue
                run_hrs   = max(MIN_RUN_HOURS, min(eff_free, headroom / r_val if r_val > 0 else eff_free))
                qty       = round(run_hrs * r_val, 0)
                p_color_val = part_color.get(p, "UNKNOWN")
                has_purge   = co_hrs > vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS) + 0.001

                machine_hours[m]     = round(machine_hours.get(m, 0) + co_hrs + run_hrs, 4)
                current_inventory[p] = round(current_inventory.get(p, 0) + qty, 0)
                machine_last_part[m] = p
                already_planned.add(p)
                remaining = round(remaining - co_hrs - run_hrs, 4)
                filled_count += 1

                plan.append({
                    "Part": p, "Color": p_color_val,
                    "Category": part_category.get(p, "?"),
                    "Fixed_Machine": part_fixed_machine.get(p, "—"),
                    "Fixed_Used": "N/A — strategic buffer", "Machine": m,
                    "Run_Hours": round(run_hrs, 3), "Changeover_Hrs": round(co_hrs, 3),
                    "Total_Hrs_Used": round(co_hrs + run_hrs, 3),
                    "Rate_Per_Hour": round(r_val, 2), "Production_Qty": qty,
                    "Indent_Daily": round(indent_daily.get(p, 0.0), 2),
                    "Demand_Daily": round(demand_daily_raw.get(p, 0.0), 2),
                    "Effective_Daily": round(daily_p, 2),
                    "Demand_Driver": demand_driver(p),
                    "Monthly_Indent": round(effective_monthly(p), 0),
                    "Daily_Indent": round(daily_p, 2),
                    "Today_Target": 0,
                    "Changeover": "No" if co_hrs == 0 else "Yes",
                    "Color_Purge": "Yes" if has_purge else "No",
                    "Type": "Strategic-Buffer",
                    "Role": "Buffer-Fill",
                    "Tools_Available": tools_available.get(p, 1),
                    "Tools_Used": 1, "Runner_Lock": "No",
                    "Priority_Score": round(strat_scores.get(p, 0), 2),
                    "Phase": 4, "Stagger_Adjusted": "No",
                })
                last_on_m  = machine_last_part.get(m)
                last_color = part_color.get(last_on_m, "UNKNOWN") if last_on_m else "UNKNOWN"
                print(f"    [SB-NEW] {p:28s} -> {m:15s}  {run_hrs:.2f}h  qty={qty:.0f}  driver={demand_driver(p)}")

        # Step C: Final extend
        remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
        if remaining > 0.001:
            _, remaining = _extend_existing_on_machine(
                m, plan, machine_hours, current_inventory,
                strat_scores, ABSOLUTE_MAX_DAYS, remaining)

    print(f"\n  Strategic buffer complete: {filled_count} extensions/additions")
    return filled_count

# =============================================================
# SECTION 21D — FORWARD LOOK  (DEMAND-AWARE)
# =============================================================

def compute_forward_look(current_inventory_after, all_parts, scenario_id):
    rows = []
    for p in all_parts:
        daily   = effective_daily(p)
        monthly = effective_monthly(p)
        if daily <= 0 or monthly <= 0:
            continue
        inv_now = current_inventory_after.get(p, 0)
        days_now = inv_now / daily if daily > 0 else 999
        days_until_safety = max(0.0, round((inv_now - SAFETY_DAYS * daily) / daily, 1))
        days_until_zero   = max(0.0, round(inv_now / daily, 1))
        alert = ""
        if days_until_zero <= FORWARD_LOOK_DAYS:
            alert = f"ZERO-STOCK RISK in {days_until_zero:.1f} days"
        elif days_until_safety <= FORWARD_LOOK_DAYS:
            alert = f"BELOW SAFETY in {days_until_safety:.1f} days"
        if alert:
            rows.append({
                "Part": p, "Color": part_color.get(p, "UNKNOWN"),
                "Category": part_category.get(p, "Stranger"),
                "Fixed_Machine": part_fixed_machine.get(p, "—"),
                "Indent_Daily": round(indent_daily.get(p, 0.0), 2),
                "Demand_Daily": round(demand_daily_raw.get(p, 0.0), 2),
                "Effective_Daily": round(daily, 2),
                "Demand_Driver": demand_driver(p),
                "Inv_After_Today": round(inv_now, 0),
                "Days_Coverage_Today": round(days_now, 2),
                "Days_Until_Safety": days_until_safety,
                "Days_Until_Zero": days_until_zero,
                "Alert": alert,
                "Action": "ESCALATE" if days_until_zero <= 2 else "Plan in next 1-3 days",
            })
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values("Days_Until_Zero").reset_index(drop=True)
    return df

# =============================================================
# SECTION 22 — OUTPUT VIEW BUILDERS  (DEMAND-AWARE)
# =============================================================

def build_multi_machine_view(plan):
    if not plan:
        return pd.DataFrame()
    from collections import defaultdict
    part_rows = defaultdict(list)
    for row in plan:
        part_rows[row["Part"]].append(row)
    multi = {p: rows for p, rows in part_rows.items() if len(rows) > 1}
    if not multi:
        return pd.DataFrame()
    output_rows = []
    for part, rows in sorted(multi.items(), key=lambda x: -sum(r["Production_Qty"] for r in x[1])):
        daily = effective_daily(part)
        total_qty = sum(float(r["Production_Qty"]) for r in rows)
        for row in rows:
            output_rows.append({
                "Part": part, "Machines_Used": len(rows), "Machine": row["Machine"],
                "Role": row.get("Role", "Primary"),
                "Run_Hours": round(float(row["Run_Hours"]), 2),
                "Production_Qty": round(float(row["Production_Qty"]), 0),
                "Effective_Daily": round(daily, 2),
                "Demand_Driver": demand_driver(part),
                "Total_Qty_All_Machines": round(total_qty, 0),
                "Type": row.get("Type", "—"),
            })
    return pd.DataFrame(output_rows)

def build_production_vs_indent(plan, all_parts):
    if not plan:
        return pd.DataFrame()
    from collections import defaultdict
    part_qty      = defaultdict(float)
    part_machines = defaultdict(list)
    for row in plan:
        p = row["Part"]
        part_qty[p]      += float(row.get("Production_Qty", 0))
        part_machines[p].append(row["Machine"])
    rows = []
    for p in sorted(part_qty.keys()):
        daily    = effective_daily(p)
        inv_b    = inventory.get(p, 0)
        produced = round(part_qty[p], 0)
        inv_after = round(inv_b + produced, 0)
        gap = round(produced - daily, 0)
        gap_dir = "OVER" if gap > 0 else ("UNDER" if gap < 0 else "MET")
        ind_d = indent_daily.get(p, 0.0)
        dem_d = demand_daily_raw.get(p, 0.0)
        demand_met = (inv_b + produced) >= (dem_d - 0.5) if dem_d > 0 else True
        indent_met = (inv_b + produced) >= (ind_d - 0.5) if ind_d > 0 else True
        rows.append({
            "Part": p, "Category": part_category.get(p, "Stranger"),
            "Machines": ", ".join(dict.fromkeys(part_machines[p])),
            "Total_Qty_Produced": produced,
            "Indent_Daily": round(ind_d, 2),
            "Demand_Daily": round(dem_d, 2),
            "Effective_Daily": round(daily, 2),
            "Demand_Driver": demand_driver(p),
            "Gap_vs_Effective_Daily": gap, "Gap_Direction": gap_dir,
            "Inventory_Before": round(inv_b, 0), "Inventory_After": inv_after,
            "Days_Coverage_After": round(inv_after / daily, 2) if daily > 0 else 0,
            "Demand_Met": "YES" if demand_met else "NO",
            "Indent_Met": "YES" if indent_met else "NO",
        })
    df = pd.DataFrame(rows)
    if not df.empty:
        order_map = {"UNDER": 0, "MET": 1, "OVER": 2}
        df["_sort"] = df["Gap_Direction"].map(order_map)
        df = df.sort_values(["_sort", "Gap_vs_Effective_Daily"]).drop(columns=["_sort"]).reset_index(drop=True)
    return df

def build_inventory_target_sheet(plan, all_parts, scenario_id):
    from collections import defaultdict
    part_produced = defaultdict(float)
    for row in plan:
        part_produced[row["Part"]] += float(row.get("Production_Qty", 0))
    rows = []
    for p in sorted(all_parts):
        daily    = effective_daily(p)
        ind_d    = indent_daily.get(p, 0.0)
        dem_d    = demand_daily_raw.get(p, 0.0)
        inv_b    = inventory.get(p, 0)
        produced = round(part_produced.get(p, 0), 0)
        inv_after = round(inv_b + produced, 0)
        days_after = round(inv_after / daily, 2) if daily > 0 else 0
        target_qty = round(TARGET_DAYS * daily, 0)
        if inv_after == 0:
            status = "CRITICAL"
        elif days_after < SAFETY_DAYS:
            status = "BELOW_SAFETY"
        elif days_after < TARGET_DAYS:
            status = "BUILDING"
        else:
            status = "AT_TARGET"
        rows.append({
            "Part": p, "Category": part_category.get(p, "Stranger"),
            "Indent_Daily": round(ind_d, 2),
            "Demand_Daily": round(dem_d, 2),
            "Effective_Daily": round(daily, 2),
            "Demand_Driver": demand_driver(p),
            "Inv_Before": round(inv_b, 0),
            "Produced_Today": produced, "Inv_After": inv_after,
            "Days_Coverage_After": days_after, "Target_Qty_5days": target_qty,
            "Buffer_Status": status,
            "Scheduled_Today": "YES" if produced > 0 else "NO",
        })
    df = pd.DataFrame(rows)
    if not df.empty:
        status_order = {"CRITICAL": 0, "BELOW_SAFETY": 1, "BUILDING": 2, "AT_TARGET": 3}
        df["_sort"] = df["Buffer_Status"].map(status_order)
        df = df.sort_values(["_sort"], ascending=True).drop(columns=["_sort"]).reset_index(drop=True)
    return df

def compute_indent_horizon(parts):
    rows = []
    for p in parts:
        inv    = inventory.get(p, 0.0)
        monthly = effective_monthly(p)
        daily  = effective_daily(p)
        r      = rate.get(p, 1.0)
        skip, skip_reason = should_skip(p)
        days_cov = inv / daily if daily > 0 else 0
        if inv == 0.0 and monthly > 0:
            status = "ZERO INV"
        elif skip and inv >= TARGET_DAYS * daily:
            status = "AT TARGET"
        elif skip:
            status = "SKIPPED"
        else:
            status = "PRODUCTION NEEDED"
        rows.append({
            "Part": p,
            "Indent_Monthly": round(indent_monthly.get(p, 0.0), 0),
            "Demand_Monthly": round(demand_monthly_raw.get(p, 0.0), 0),
            "Effective_Monthly": round(monthly, 0),
            "Indent_Daily": round(indent_daily.get(p, 0.0), 2),
            "Demand_Daily": round(demand_daily_raw.get(p, 0.0), 2),
            "Effective_Daily": round(daily, 2),
            "Demand_Driver": demand_driver(p),
            "Inventory_Now": round(inv, 0),
            "Days_Coverage": round(days_cov, 2),
            "Indent_Status": status,
            "Skip_Reason": skip_reason,
        })
    return pd.DataFrame(rows)

def build_terminal_status_sheet():
    rows = []
    all_terminals_known = set(terminal_status.keys())
    for terms in part_terminals.values():
        for t in terms:
            all_terminals_known.add(t)
    for t in sorted(all_terminals_known):
        inv = terminal_status.get(t, None)
        parts_needing = [p for p, terms in part_terminals.items() if t in terms]
        parts_blocked = []
        for p in parts_needing:
            cat   = part_category.get(p, "Stranger")
            daily = effective_daily(p)
            mult  = TERMINAL_THRESHOLD.get(cat, 0.25)
            thresh = mult * daily
            if inv is None or inv < thresh:
                parts_blocked.append(f"{p}(need>={thresh:.0f})")
        rows.append({
            "Terminal": t,
            "Inventory": round(inv, 0) if inv is not None else "NOT IN SHEET",
            "Parts_Requiring": ", ".join(sorted(parts_needing)) if parts_needing else "—",
            "Parts_Blocked": ", ".join(parts_blocked) if parts_blocked else "—",
            "Impact": "BLOCKING" if parts_blocked else ("Adequate" if parts_needing else "No parts"),
        })
    return pd.DataFrame(rows)

def build_fixed_machine_status(plan, current_inventory):
    rows = []
    for machine, fixed_parts in sorted(machine_fixed_parts.items()):
        phase    = "A" if machine in _phase_a_machines else "B"
        hrs_used = sum(float(r.get("Run_Hours", 0)) + float(r.get("Changeover_Hrs", 0))
                       for r in plan if r["Machine"] == machine)
        for p in fixed_parts:
            daily  = effective_daily(p)
            inv_b  = inventory.get(p, 0)
            inv_now = current_inventory.get(p, inv_b)
            days_now = round(inv_now / daily, 2) if daily > 0 else 0
            produced = round(inv_now - inv_b, 0)
            rows.append({
                "Machine": machine, "Phase": f"Phase {phase}",
                "Part": p,
                "Indent_Daily": round(indent_daily.get(p, 0.0), 2),
                "Demand_Daily": round(demand_daily_raw.get(p, 0.0), 2),
                "Effective_Daily": round(daily, 2),
                "Demand_Driver": demand_driver(p),
                "Inv_Before": round(inv_b, 0), "Produced_Today": produced,
                "Inv_After": round(inv_now, 0), "Days_Coverage_After": days_now,
                "Machine_Hrs_Used": round(hrs_used, 2),
            })
    return pd.DataFrame(rows)

SPECIALIZED_MACHINE_THRESHOLD = 3

def detect_specialized_machines(all_parts):
    machine_parts = {}
    for m in vt_machines:
        machine_parts[m] = [
            p for p in all_parts
            if m in vt_compat.get(p, []) and not should_skip(p)[0] and effective_monthly(p) > 0
        ]
    specialized = {m for m, mp in machine_parts.items() if 0 < len(mp) <= SPECIALIZED_MACHINE_THRESHOLD}
    return specialized, {m: machine_parts[m] for m in specialized}, machine_parts

# =============================================================
# SECTION 23 — MAIN SCHEDULER  (DEMAND-AWARE)
# =============================================================

def schedule(parts, label=""):
    global _phase_a_machines
    print(f"\n{'─'*65}")
    print(f"  {label}  |  {len(parts)} parts  |  {len(vt_machines)} machines")
    print(f"{'─'*65}")

    scenario_id, scenario_desc = classify_scenario(parts)
    print(f"  {scenario_desc}")
    print(f"  OPD cap: {opd_cap(scenario_id)} days")

    horizon_df = compute_indent_horizon(parts)
    active_parts = [
        p for p in parts
        if not is_scheduling_skip(p) and effective_daily(p) > 0
    ]
    priority_scores, score_rows = compute_priority_scores(active_parts)
    score_df = pd.DataFrame(score_rows) if score_rows else pd.DataFrame()

    machine_hours      = {m: 0.0 for m in vt_machines}
    machine_last_part  = {m: machine_state.get(m) for m in vt_machines}
    current_inventory  = inventory.copy()
    inventory_start_of_day = inventory.copy()

    plan           = []
    already_planned = set()
    not_planned    = []
    deferred       = []

    _phase_a_machines, _ = schedule_fixed_machines(
        machine_hours, machine_last_part, current_inventory,
        plan, already_planned, priority_scores, scenario_id)

    specialized_machines, spec_machine_parts, _ = detect_specialized_machines(list(parts))

    zero_inv_rr = [
        p for p in active_parts
        if current_inventory.get(p, 0) == 0 and p not in already_planned
        and part_category.get(p, "Stranger") in ("Runner", "Repeater") and vt_compat.get(p)
    ]

    sorted_active = sorted(
        [p for p in active_parts if p not in already_planned],
        key=lambda p: priority_scores.get(p, 0), reverse=True,
    )

    print(f"\n  PRIMARY PASS  ({len(sorted_active)} parts)")
    for part in sorted_active:
        monthly = effective_monthly(part)

        if monthly == 0:
            deferred.append({"Part": part, "Reason": "Effective monthly = 0 (no indent, no demand)"})
            continue

        t_blocked, t_reason = terminal_blocked(part)
        if t_blocked:
            not_planned.append({"Part": part, "Reason": t_reason})
            continue

        if not vt_compat.get(part):
            not_planned.append({"Part": part, "Reason": "Not in VT_Matrix"})
            continue

        new_rows = assign_part(part, scenario_id, machine_hours, machine_last_part,
                               current_inventory, plan, already_planned, priority_scores)
        if new_rows:
            plan.extend(new_rows)
            total_qty   = sum(float(r["Production_Qty"]) for r in new_rows)
            machines_str = ", ".join([r["Machine"] for r in new_rows])
            print(f"  {part:<30} -> {machines_str:<25}  qty={total_qty:.0f}  driver={demand_driver(part)}")
        else:
            not_planned.append({"Part": part, "Reason": "No capacity"})

    unscheduled_zero = [p for p in zero_inv_rr if p not in already_planned]
    for part in unscheduled_zero:
        success = displace_for_zero_inv(part, machine_hours, machine_last_part,
                                         current_inventory, plan, already_planned, priority_scores)
        if success:
            not_planned[:] = [r for r in not_planned if r.get("Part") != part]

    runner_priority_log = enforce_runner_priority(
        plan, machine_hours, machine_last_part, current_inventory,
        already_planned, priority_scores, inventory_start_of_day)
    newly_planned = {r["Runner_Part"] for r in runner_priority_log if "PLANNED" in r.get("Result", "")}
    not_planned[:] = [r for r in not_planned if r.get("Part") not in newly_planned]

    micro_idle = utilization_enforcer(
        plan, machine_hours, machine_last_part, list(parts),
        already_planned, current_inventory, scenario_id, priority_scores)

    strategic_buffer_filler(plan, machine_hours, machine_last_part,
                            list(parts), already_planned, current_inventory, scenario_id)

    reconcile_machine_hours(plan, machine_hours)
    plan = resequence_machine_rows(plan, machine_state)
    reconcile_machine_hours(plan, machine_hours)

    for m in vt_machines:
        m_rows = [r for r in plan if r["Machine"] == m]
        if m_rows:
            machine_last_part[m] = m_rows[-1]["Part"]

    stagger_co_by_quantity(plan, vt_machines, scenario_id, current_inventory)
    stagger_changeovers(plan, vt_machines, machine_hours)
    reconcile_machine_hours(plan, machine_hours)

    violations = validate_plan_rows(plan, current_inventory)

    forward_look_df      = compute_forward_look(current_inventory, list(parts), scenario_id)
    multi_machine_df     = build_multi_machine_view(plan)
    prod_vs_indent_df    = build_production_vs_indent(plan, list(parts))
    inv_target_df        = build_inventory_target_sheet(plan, list(parts), scenario_id)
    runner_priority_df   = pd.DataFrame(runner_priority_log) if runner_priority_log else pd.DataFrame()
    fixed_machine_status_df = build_fixed_machine_status(plan, current_inventory)
    violations_df        = pd.DataFrame(violations) if violations else pd.DataFrame()

    mach_rows = []
    for m in vt_machines:
        used      = machine_hours.get(m, 0)
        parts_run = list({r["Part"] for r in plan if r["Machine"] == m})
        co_count  = sum(1 for r in plan if r["Machine"] == m and r.get("Changeover") == "Yes")
        util_pct  = round(used / AVAILABLE_HOURS * 100, 1)
        mach_rows.append({
            "Machine": m, "Used_Hours": round(used, 2),
            "Unused_Hours": round(AVAILABLE_HOURS - used, 2),
            "Utilization_%": util_pct, "CO_Count": co_count,
            "Status": ("FULL"     if used >= AVAILABLE_HOURS - 0.3 else
                       "GOOD"     if util_pct >= 98 else
                       "OK"       if util_pct >= 90 else "UNDERUSED"),
            "Parts_Planned": len(parts_run),
            "Last_Part": machine_last_part.get(m) or "—",
            "All_Parts": ", ".join(parts_run) if parts_run else "— idle —",
        })

    inv_rows = []
    for p in parts:
        ind_d    = indent_daily.get(p, 0.0)
        dem_d    = demand_daily_raw.get(p, 0.0)
        daily    = effective_daily(p)
        inv_b    = inventory.get(p, 0)
        produced = sum(float(r["Production_Qty"]) for r in plan if r["Part"] == p)
        inv_after = inv_b + produced
        days_cov  = inv_after / daily if daily > 0 else 0
        demand_ok = (inv_after >= dem_d - 0.5) if dem_d > 0 else True
        inv_rows.append({
            "Part": p,
            "Indent_Daily": round(ind_d, 2),
            "Demand_Daily": round(dem_d, 2),
            "Effective_Daily": round(daily, 2),
            "Demand_Driver": demand_driver(p),
            "Inv_Before": round(inv_b, 0), "Produced_Today": round(produced, 0),
            "Inv_After": round(inv_after, 0), "Days_Coverage": round(days_cov, 2),
            "Demand_Met_Today": "YES" if demand_ok else "NO",
            "Status": ("AT_TARGET" if days_cov >= TARGET_DAYS else
                       "OK"        if days_cov >= SAFETY_DAYS else
                       "LOW"       if days_cov >= 1 else "CRITICAL"),
        })

    micro_df  = pd.DataFrame(micro_idle) if micro_idle else pd.DataFrame()
    plan_df   = pd.DataFrame(plan) if plan else pd.DataFrame()
    def_df    = pd.DataFrame(deferred) if deferred else pd.DataFrame()
    not_df    = pd.DataFrame(not_planned) if not_planned else pd.DataFrame()
    mach_df   = pd.DataFrame(mach_rows)
    inv_df    = pd.DataFrame(inv_rows)

    if not plan_df.empty:
        plan_df.insert(0, "Planning_Date", str(PLANNING_DATE))
        plan_df.insert(1, "Scenario", scenario_desc)

    # ── Demand coverage summary ───────────────────────────────
    demand_parts = [p for p in parts if demand_daily_raw.get(p, 0) > 0]
    demand_met_count = 0
    demand_not_met   = []
    for p in demand_parts:
        inv_b    = inventory.get(p, 0)
        produced = sum(float(r.get("Production_Qty", 0)) for r in plan if r["Part"] == p)
        dem_d    = demand_daily_raw[p]
        if inv_b + produced >= dem_d - 0.5:
            demand_met_count += 1
        else:
            demand_not_met.append(p)

    print(f"\n  {'='*65}")
    print(f"  SUMMARY — {scenario_desc}")
    print(f"    Parts planned  : {len(already_planned)}")
    print(f"    Not planned    : {len(not_planned)}")
    print(f"    Deferred       : {len(deferred)}")
    print(f"    Violations     : {len(violations)}")
    if demand_parts:
        print(f"    Demand parts   : {len(demand_parts)}")
        print(f"    Demand MET     : {demand_met_count}/{len(demand_parts)}")
        if demand_not_met:
            print(f"    Demand NOT MET : {', '.join(demand_not_met)}")
    if not mach_df.empty:
        print(f"    Avg utilization: {mach_df['Utilization_%'].mean():.1f}%")
        underused = (mach_df['Status'] == 'UNDERUSED').sum()
        if underused > 0:
            print(f"    UNDERUSED      : {underused} machines")
    print(f"  {'='*65}")

    return (plan_df, def_df, not_df, mach_df, inv_df, machine_last_part,
            horizon_df, pd.DataFrame(), score_df, micro_df, multi_machine_df,
            prod_vs_indent_df, inv_target_df, runner_priority_df,
            fixed_machine_status_df, forward_look_df, violations_df)

# =============================================================
# SECTION 24 — PART AUDIT  (DEMAND-AWARE)
# =============================================================

all_book_parts      = list(data["Material"].unique())
all_demand_parts    = list(demand_daily_raw.keys())
all_universe_parts  = list(dict.fromkeys(all_book_parts + all_demand_parts))

matrix_parts  = set(str(p).strip() for p in vt_matrix["Part"] if pd.notna(p))
zero_rate_set = set(data_zero_rate["Material"].unique())

for p in all_demand_parts:
    if p not in inventory:
        inventory[p] = 0.0
    if p not in part_color:
        part_color[p]  = "UNKNOWN"
        ALL_KNOWN_COLORS[p] = "UNKNOWN"

vt_parts = data_valid[data_valid["Material"].isin(vt_matrix["Part"])]["Material"].unique()

audit_rows = []
for part in all_universe_parts:
    inv    = inventory.get(part, 0.0)
    r_val  = rate.get(part, None)
    monthly = effective_monthly(part)
    ind_d  = indent_daily.get(part, 0.0)
    dem_d  = demand_daily_raw.get(part, 0.0)
    daily  = effective_daily(part)
    days_cov = inv / daily if daily > 0 else 0
    t_blk, t_rsn = terminal_blocked(part)

    if part in zero_rate_set or r_val is None:
        status = "ZERO/MISSING CYCLE TIME"
    elif part not in matrix_parts:
        status = "NOT IN VT_MATRIX"
    elif monthly == 0:
        status = "ZERO INDENT+DEMAND"
    elif daily <= MIN_DAILY_INDENT and dem_d <= 0:
        status = "SKIPPED (LOW INDENT, NO DEMAND)"
    elif daily > 0 and inv >= TARGET_DAYS * daily:
        status = "AT TARGET — SKIP"
    elif t_blk:
        status = "BLOCKED — TERMINAL"
    else:
        status = "ENTERS SCHEDULER"

    audit_rows.append({
        "Part": part, "Color": part_color.get(part, "UNKNOWN"),
        "Indent_Monthly": round(indent_monthly.get(part, 0.0), 0),
        "Demand_Monthly": round(demand_monthly_raw.get(part, 0.0), 0),
        "Indent_Daily": round(ind_d, 2),
        "Demand_Daily": round(dem_d, 2),
        "Effective_Daily": round(daily, 2),
        "Demand_Driver": demand_driver(part),
        "Inventory": round(inv, 0), "Days_Coverage": round(days_cov, 2),
        "Rate_Per_Hour": round(r_val, 2) if r_val else "—",
        "Status": status,
    })

audit_df = pd.DataFrame(audit_rows)
print(f"\n  Part audit ({len(all_universe_parts)} total):")
for status, count in audit_df["Status"].value_counts().items():
    marker = "+" if status == "ENTERS SCHEDULER" else "·"
    print(f"    {marker}  {status:<55}: {count:>4}")

vt_terminal_status_df = build_terminal_status_sheet()

# =============================================================
# SECTION 25 — RUN
# =============================================================

schedulable_parts = [
    p for p in all_universe_parts
    if p in matrix_parts and (rate.get(p, 0) or 0) > 0
]

(vt_plan, vt_def, vt_not, vt_mach, vt_inv,
 vt_state, vt_horizon, vt_indent_status,
 vt_scores, vt_micro, vt_multi_machine,
 vt_prod_vs_indent, vt_inv_target,
 vt_runner_priority, vt_fixed_status,
 vt_forward_look, vt_violations) = schedule(schedulable_parts, "VT Machines")

save_machine_state(vt_state)

# =============================================================
# SECTION 26 — MACHINE-WISE PLAN
# =============================================================

def build_machine_wise_plan(plan_df):
    if plan_df.empty:
        return pd.DataFrame()
    rows = []
    for m in vt_machines:
        machine_rows = plan_df[plan_df["Machine"] == m].copy()
        if machine_rows.empty:
            continue
        for _, pr in machine_rows.iterrows():
            p = pr.get("Part", "—")
            rows.append({
                "Machine": m, "Part": p,
                "Color": part_color.get(p, "UNKNOWN"),
                "Category": part_category.get(p, "Stranger"),
                "Role": pr.get("Role", "Primary"),
                "Run_Hours": round(float(pr.get("Run_Hours", 0) or 0), 2),
                "Changeover_Hrs": round(float(pr.get("Changeover_Hrs", 0) or 0), 3),
                "Production_Qty": round(float(pr.get("Production_Qty", 0) or 0), 0),
                "Indent_Daily": round(float(pr.get("Indent_Daily", 0) or 0), 2),
                "Demand_Daily": round(float(pr.get("Demand_Daily", 0) or 0), 2),
                "Effective_Daily": round(float(pr.get("Effective_Daily", 0) or 0), 2),
                "Demand_Driver": pr.get("Demand_Driver", "—"),
                "Type": pr.get("Type", "Primary") or "Primary",
                "Row_Type": "Part",
            })
        co_total  = machine_rows["Changeover_Hrs"].apply(lambda x: float(x) if pd.notna(x) else 0).sum()
        run_total = machine_rows["Run_Hours"].apply(lambda x: float(x) if pd.notna(x) else 0).sum()
        qty_total = machine_rows["Production_Qty"].apply(lambda x: float(x) if pd.notna(x) else 0).sum()
        hrs_total = round(co_total + run_total, 2)
        rows.append({
            "Machine": m, "Part": f"TOTAL — {m}",
            "Color": "—", "Category": "—", "Role": "—",
            "Run_Hours": round(run_total, 2),
            "Changeover_Hrs": round(co_total, 2),
            "Production_Qty": round(qty_total, 0),
            "Indent_Daily": "—", "Demand_Daily": "—", "Effective_Daily": "—",
            "Demand_Driver": "—",
            "Type": f"Total {hrs_total}h / {AVAILABLE_HOURS}h  |  Util {round(hrs_total/AVAILABLE_HOURS*100,1)}%",
            "Row_Type": "Summary",
        })
        rows.append({k: "" for k in rows[-1].keys()})
    return pd.DataFrame(rows)

def build_co_queue(plan, machines):
    events = _collect_co_events(plan, machines)
    if not events:
        return pd.DataFrame()
    events.sort(key=lambda e: e["natural_start"])
    rows = []
    tool_changer_free_at = 0.0
    for pos, ev in enumerate(events, 1):
        natural_start = _recompute_natural_start(ev, plan)
        co_h          = ev["co_duration"]
        actual_start  = max(natural_start, tool_changer_free_at)
        tool_changer_free_at = actual_start + co_h
        before_color = part_color.get(ev["part_before"], "UNKNOWN")
        after_color  = part_color.get(ev["part_after"],  "UNKNOWN")
        color_change = before_color != after_color and before_color != "UNKNOWN" and after_color != "UNKNOWN"
        rows.append({
            "Queue_Position": pos, "Machine": ev["machine"],
            "Part_Before": ev["part_before"], "Part_After": ev["part_after"],
            "Color_Before": before_color, "Color_After": after_color,
            "Color_Change": "YES — PURGE" if color_change else "No",
            "CO_Duration_Min": round(co_h * 60, 1),
        })
    return pd.DataFrame(rows)

# =============================================================
# SECTION 27 — EXCEL OUTPUT
# =============================================================

from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

HEADER_COLORS = {
    "VT_Plan_By_Machine":       "0D6E6E",
    "VT_CO_Queue":              "375623",
    "VT_Plan":                  "1F4E79",
    "VT_Multi_Machine_Parts":   "4A235A",
    "VT_Production_vs_Indent":  "154360",
    "VT_Inventory_Target":      "1B4F72",
    "VT_Priority_Scores":       "2C4770",
    "VT_Machine_Util":          "375623",
    "VT_Not_Planned":           "7B2C2C",
    "VT_Deferred":              "7F6000",
    "VT_Inventory_Health":      "4A235A",
    "VT_Indent_Horizon":        "154360",
    "VT_Part_Audit":            "1C3557",
    "VT_Micro_Idle":            "5C3D2E",
    "VT_Terminal_Status":       "7B1C1C",
    "VT_Runner_Priority_Log":   "7B3F00",
    "VT_Fixed_Machine_Status":  "1A5276",
    "VT_Forward_Look":          "6D28D9",
    "VT_Violations":            "991B1B",
}

STATUS_FILLS = {
    "FULL":          PatternFill("solid", fgColor="C6EFCE"),
    "GOOD":          PatternFill("solid", fgColor="DDEBF7"),
    "OK":            PatternFill("solid", fgColor="EBF5E1"),
    "UNDERUSED":     PatternFill("solid", fgColor="FFC7CE"),
    "AT_TARGET":     PatternFill("solid", fgColor="C6EFCE"),
    "BUILDING":      PatternFill("solid", fgColor="DDEBF7"),
    "BELOW_SAFETY":  PatternFill("solid", fgColor="FFEB9C"),
    "CRITICAL":      PatternFill("solid", fgColor="FFC7CE"),
    "LOW":           PatternFill("solid", fgColor="FFEB9C"),
    "OVER":          PatternFill("solid", fgColor="DDEBF7"),
    "UNDER":         PatternFill("solid", fgColor="FFC7CE"),
    "MET":           PatternFill("solid", fgColor="C6EFCE"),
    "YES":           PatternFill("solid", fgColor="C6EFCE"),
    "NO":            PatternFill("solid", fgColor="FFC7CE"),
    "YES — PURGE":   PatternFill("solid", fgColor="FFC7CE"),
    "BLOCKING":      PatternFill("solid", fgColor="FFC7CE"),
    "Adequate":      PatternFill("solid", fgColor="C6EFCE"),
    "ENTERS SCHEDULER": PatternFill("solid", fgColor="C6EFCE"),
}

def style_sheet(ws, header_hex):
    for cell in ws[1]:
        cell.fill      = PatternFill("solid", fgColor=header_hex)
        cell.font      = Font(bold=True, color="FFFFFF", size=11)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[1].height = 32
    for col in ws.columns:
        col_letter = get_column_letter(col[0].column)
        max_len    = max((len(str(c.value)) for c in col if c.value is not None), default=8)
        ws.column_dimensions[col_letter].width = max(10, min(55, max_len + 3))
    headers = [cell.value for cell in ws[1]]
    status_cols = ["Status", "Gap_Direction", "Buffer_Status", "Color_Change",
                   "Impact", "Result", "Demand_Met", "Demand_Met_Today",
                   "Indent_Met", "Demand_Driver"]
    for col_idx, col_name in enumerate(headers, start=1):
        if col_name and any(x in str(col_name) for x in status_cols):
            for row in ws.iter_rows(min_row=2, min_col=col_idx, max_col=col_idx):
                for cell in row:
                    fill = STATUS_FILLS.get(str(cell.value))
                    if fill:
                        cell.fill = fill
    ws.freeze_panes = "A2"

def style_machine_wise_sheet(ws):
    header_fill  = PatternFill("solid", fgColor="1F4E79")
    summary_fill = PatternFill("solid", fgColor="0D9488")
    part_fills   = [PatternFill("solid", fgColor="EFF6FF"), PatternFill("solid", fgColor="F0FDF4")]
    for cell in ws[1]:
        cell.fill      = header_fill
        cell.font      = Font(bold=True, color="FFFFFF", size=11)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[1].height = 30
    headers       = [cell.value for cell in ws[1]]
    row_type_col  = headers.index("Row_Type") + 1 if "Row_Type" in headers else None
    machine_col   = headers.index("Machine")  + 1 if "Machine"  in headers else None
    machine_color_idx = 0
    current_machine   = None
    for row in ws.iter_rows(min_row=2):
        row_type = row[row_type_col - 1].value if row_type_col else ""
        machine  = row[machine_col  - 1].value if machine_col  else ""
        if machine and machine != current_machine:
            current_machine   = machine
            machine_color_idx = (machine_color_idx + 1) % 2
        if row_type == "Summary":
            for cell in row:
                cell.fill = summary_fill
                cell.font = Font(bold=True, color="FFFFFF", size=11)
        elif row_type == "Part":
            for cell in row:
                cell.fill = part_fills[machine_color_idx]
    for col in ws.columns:
        col_letter = get_column_letter(col[0].column)
        max_len    = max((len(str(c.value)) for c in col if c.value), default=8)
        ws.column_dimensions[col_letter].width = max(10, min(45, max_len + 3))
    ws.freeze_panes = "B2"

print(f"\nWriting output -> {output_path}")

vt_mw  = build_machine_wise_plan(vt_plan)
vt_co_q = build_co_queue(
    [{k: v for k, v in r.items()} for r in vt_plan.to_dict("records")] if not vt_plan.empty else [],
    vt_machines,
)

sheets = {
    "VT_Plan_By_Machine":       vt_mw,
    "VT_CO_Queue":              vt_co_q,
    "VT_Plan":                  vt_plan,
    "VT_Fixed_Machine_Status":  vt_fixed_status,
    "VT_Runner_Priority_Log":   vt_runner_priority,
    "VT_Inventory_Target":      vt_inv_target,
    "VT_Forward_Look":          vt_forward_look,
    "VT_Multi_Machine_Parts":   vt_multi_machine,
    "VT_Production_vs_Indent":  vt_prod_vs_indent,
    "VT_Priority_Scores":       vt_scores,
    "VT_Machine_Util":          vt_mach,
    "VT_Not_Planned":           vt_not,
    "VT_Deferred":              vt_def,
    "VT_Inventory_Health":      vt_inv,
    "VT_Indent_Horizon":        vt_horizon,
    "VT_Part_Audit":            audit_df,
    "VT_Terminal_Status":       vt_terminal_status_df,
}
if not vt_micro.empty:
    sheets["VT_Micro_Idle"] = vt_micro
if not vt_violations.empty:
    sheets["VT_Violations"] = vt_violations

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for sheet_name, df in sheets.items():
        if df is not None and not df.empty:
            df.to_excel(writer, sheet_name=sheet_name, index=False)

wb = load_workbook(output_path)

if "VT_Plan_By_Machine" in wb.sheetnames:
    style_machine_wise_sheet(wb["VT_Plan_By_Machine"])

for sheet_name, header_hex in HEADER_COLORS.items():
    if sheet_name in wb.sheetnames and sheet_name != "VT_Plan_By_Machine":
        style_sheet(wb[sheet_name], header_hex)

for name, color in HEADER_COLORS.items():
    if name in wb.sheetnames:
        wb[name].sheet_properties.tabColor = color

wb.save(output_path)
print(f"  Formatting applied")

# =============================================================
# SECTION 28 — FINAL SUMMARY
# =============================================================

print(f"\n{'='*65}")
print(f"  Smart APS V11-DEMAND  —  {PLANNING_DATE}")
print(f"  Safety: {SAFETY_DAYS}d  |  Target: {TARGET_DAYS}d  |  Max CO: {MAX_DAILY_CO}")
print(f"{'='*65}")

for status, count in audit_df["Status"].value_counts().items():
    marker = "+" if status == "ENTERS SCHEDULER" else "·"
    print(f"  {marker} {status:<55}: {count:>4}")

print(f"\n  Results:")
print(f"    Plan rows     : {len(vt_plan):>4}")
print(f"    Not planned   : {len(vt_not):>4}")
print(f"    Deferred      : {len(vt_def):>4}")
print(f"    Violations    : {len(vt_violations):>4}")

if not vt_mach.empty:
    print(f"\n  Machine utilization:")
    print(f"    Average       : {vt_mach['Utilization_%'].mean():.1f}%")
    underused = (vt_mach['Status'] == 'UNDERUSED').sum()
    if underused > 0:
        print(f"    UNDERUSED     : {underused} machines")

if not vt_inv_target.empty:
    at_t = (vt_inv_target["Buffer_Status"] == "AT_TARGET").sum()
    bld  = (vt_inv_target["Buffer_Status"] == "BUILDING").sum()
    bls  = (vt_inv_target["Buffer_Status"] == "BELOW_SAFETY").sum()
    crt  = (vt_inv_target["Buffer_Status"] == "CRITICAL").sum()
    print(f"\n  Inventory after today:")
    print(f"    AT_TARGET  : {at_t:>4}")
    print(f"    BUILDING   : {bld:>4}")
    print(f"    BELOW_SAFE : {bls:>4}")
    print(f"    CRITICAL   : {crt:>4}")

if not vt_prod_vs_indent.empty and "Demand_Met" in vt_prod_vs_indent.columns:
    dm_yes = (vt_prod_vs_indent["Demand_Met"] == "YES").sum()
    dm_no  = (vt_prod_vs_indent["Demand_Met"] == "NO").sum()
    print(f"\n  Demand coverage:")
    print(f"    Demand MET     : {dm_yes:>4}")
    print(f"    Demand NOT MET : {dm_no:>4}")

print(f"\n  Output -> {output_path}")
print(f"  State  -> {MACHINE_STATE_FILE}")
print(f"\n  UPDATE DAILY: PLANNING_DATE = date(2026, 4, 10)")
print(f"{'='*65}")